# Library-free replication: sequential C (CPU) vs hand-written CUDA (GPU), Kaggle T4

Same experiments as the PyTorch notebooks (paper section 3: conv layer vs batch, FC matvec vs n, plus the bandwidth
checks), but every kernel is written by hand: `bench_seq` is plain C on one thread (no BLAS, no OpenMP),
`bench_cuda` uses only the CUDA runtime (no cuBLAS, no cuDNN). 3 runs, GPU first (30 repeats per point),
then CPU (10 repeats). Estimated 30–45 minutes (not measured yet).

**Before running**
1. Settings → Accelerator → **GPU T4 x2**.
2. Run All. A failing build or correctness check stops the notebook before anything is timed.
3. Download `results_c_cuda.zip` from the Output panel, then stop the session.

The plan and the predictions are in `PLAN_C_CUDA.md` in the repository, written before this notebook was run.
The cells below are generated from `c_cuda/` and `code/gpu_sampler.py` by `code/make_c_cuda_notebook.py`.

## 1 · Machine and tools

In [ ]:
import os
os.environ["PATH"] = os.environ.get("PATH", "") + ":/usr/local/cuda/bin"
!nvidia-smi
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core|Socket|L1d|L2|L3|Flags' | cut -c1-300
!gcc --version | head -1
!nvcc --version | tail -2
!mkdir -p /kaggle/working/c_cuda

## 2 · Sources (written to `/kaggle/working/c_cuda`)

In [ ]:
%%writefile /kaggle/working/c_cuda/common.h
/* common.h - shared by bench_seq.c (sequential C) and the host side of bench_cuda.cu (CUDA).

   Plain C, no threads, no BLAS, no external library: only the C standard library, plus the
   POSIX (or Win32) calls for the clocks and mkdir. Compiles as C11 and as C++ (nvcc host code).

   Contents: layer constants (the paper's first conv layer), clocks, random data, statistics,
   CSV/JSON output, command-line parsing, and the float64 reference kernels used by --check. */
#ifndef HPC_COMMON_H
#define HPC_COMMON_H

#if !defined(_WIN32) && !defined(_POSIX_C_SOURCE)
#define _POSIX_C_SOURCE 200809L
#endif

#include <math.h>
#include <stdarg.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#ifdef _WIN32
#ifndef _WIN32_WINNT
#define _WIN32_WINNT 0x0A00
#endif
#include <direct.h>
#include <windows.h>
#else
#include <sys/stat.h>
#include <sys/types.h>
#include <unistd.h>
#endif

#if defined(_MSC_VER)
#define RESTRICT __restrict
#elif defined(__cplusplus)
#define RESTRICT __restrict__
#else
#define RESTRICT restrict
#endif

#if defined(__GNUC__)
#define MAYBE_UNUSED __attribute__((unused))
#define NOINLINE __attribute__((noinline))
#define NORETURN __attribute__((noreturn))
#elif defined(_MSC_VER)
#define MAYBE_UNUSED
#define NOINLINE __declspec(noinline)
#define NORETURN __declspec(noreturn)
#else
#define MAYBE_UNUSED
#define NOINLINE
#define NORETURN
#endif
#define API static MAYBE_UNUSED

/* ------------------------------------------------------------------ the paper's conv layer
   Greyscale 224x224 input, 96 filters of 11x11, stride 4, no padding: output 96 x 54 x 54.
   Caffe unrolls each image (im2col) into a 121 x 2916 matrix and multiplies W (96 x 121) by it. */
#define IMG 224
#define C_IN 1
#define C_OUT 96
#define KS 11
#define STRIDE 4
#define OUT_HW ((IMG - KS) / STRIDE + 1)                  /* 54 */
#define PATCHES (OUT_HW * OUT_HW)                         /* 2916 */
#define PATCH_LEN (C_IN * KS * KS)                        /* 121 */
#define IMG_ELEMS ((long)C_IN * IMG * IMG)
#define OUT_ELEMS ((long)C_OUT * PATCHES)
#define COL_ELEMS ((long)PATCH_LEN * PATCHES)
#define FLOPS_PER_IMAGE (2.0 * C_OUT * PATCH_LEN * PATCHES) /* multiply-adds of the GEMM, x2 */

#define MAX_LIST 64

/* ------------------------------------------------------------------ logging */
static FILE *g_log = NULL;

API void say(const char *fmt, ...) {
    va_list ap;
    va_start(ap, fmt);
    vprintf(fmt, ap);
    va_end(ap);
    printf("\n");
    fflush(stdout);
    if (g_log) {
        va_start(ap, fmt);
        vfprintf(g_log, fmt, ap);
        va_end(ap);
        fputc('\n', g_log);
        fflush(g_log);
    }
}

API NORETURN void die(const char *fmt, ...) {
    va_list ap;
    va_start(ap, fmt);
    fprintf(stderr, "error: ");
    vfprintf(stderr, fmt, ap);
    va_end(ap);
    fprintf(stderr, "\n");
    exit(1);
}

/* ------------------------------------------------------------------ clocks */
/* Monotonic seconds, for measuring durations. */
API double now_s(void) {
#ifdef _WIN32
    static LARGE_INTEGER freq;
    LARGE_INTEGER c;
    if (!freq.QuadPart) QueryPerformanceFrequency(&freq);
    QueryPerformanceCounter(&c);
    return (double)c.QuadPart / (double)freq.QuadPart;
#else
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + 1e-9 * (double)ts.tv_nsec;
#endif
}

/* Seconds since 1970 (same clock as Python's time.time()), to line rows up with gpu_clocks.csv. */
API double epoch_s(void) {
#ifdef _WIN32
    FILETIME ft;
    ULARGE_INTEGER u;
    GetSystemTimePreciseAsFileTime(&ft);
    u.LowPart = ft.dwLowDateTime;
    u.HighPart = ft.dwHighDateTime;
    return (double)(u.QuadPart - 116444736000000000ULL) * 1e-7;
#else
    struct timespec ts;
    clock_gettime(CLOCK_REALTIME, &ts);
    return (double)ts.tv_sec + 1e-9 * (double)ts.tv_nsec;
#endif
}

API void iso_utc(char *buf, size_t n) {
    time_t t = time(NULL);
    struct tm g;
#ifdef _WIN32
    gmtime_s(&g, &t);
#else
    gmtime_r(&t, &g);
#endif
    strftime(buf, n, "%Y-%m-%dT%H:%M:%S+00:00", &g);
}

/* ------------------------------------------------------------------ random data (splitmix64) */
API unsigned long long splitmix64(unsigned long long *s) {
    unsigned long long z = (*s += 0x9E3779B97F4A7C15ULL);
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
    return z ^ (z >> 31);
}

static unsigned long long g_rng = 1;
API void rng_seed(unsigned long long seed) { g_rng = seed * 0x2545F4914F6CDD1DULL + 12345; }
/* Uniform in [-1, 1), 24 random bits. */
API float rng_float(void) { return (float)((double)(splitmix64(&g_rng) >> 40) * (2.0 / 16777216.0) - 1.0); }
API void fill_random(float *p, long n) {
    for (long i = 0; i < n; i++) p[i] = rng_float();
}

/* Fisher-Yates with its own generator. Seed 0 keeps the list as given (ascending). */
API void shuffle(long *a, int n, int seed) {
    unsigned long long s = 0x853C49E6748FEA9BULL ^ ((unsigned long long)seed * 0x9E3779B97F4A7C15ULL);
    if (!seed) return;
    for (int i = n - 1; i > 0; i--) {
        int j = (int)(splitmix64(&s) % (unsigned long long)(i + 1));
        long t = a[i];
        a[i] = a[j];
        a[j] = t;
    }
}

/* ------------------------------------------------------------------ memory */
API float *falloc(long n) {
    size_t bytes = (size_t)(n > 0 ? n : 1) * sizeof(float);
    void *p = NULL;
#ifdef _WIN32
    p = _aligned_malloc(bytes, 64);
#else
    if (posix_memalign(&p, 64, bytes)) p = NULL;
#endif
    if (!p) die("out of memory (%zu bytes)", bytes);
    return (float *)p;
}

API void ffree(float *p) {
#ifdef _WIN32
    _aligned_free(p);
#else
    free(p);
#endif
}

API void fzero(float *p, long n) { memset(p, 0, (size_t)n * sizeof(float)); }

API double *dalloc(long n) {
    double *p = (double *)malloc((size_t)(n > 0 ? n : 1) * sizeof(double));
    if (!p) die("out of memory");
    return p;
}

/* ------------------------------------------------------------------ statistics */
typedef struct {
    double mean, std, min, median, issue, t_start, t_end;
} Stats;

API int cmp_double(const void *a, const void *b) {
    double x = *(const double *)a, y = *(const double *)b;
    return (x > y) - (x < y);
}

/* ms[i]: wall time of timed run i; issue[i]: time until the host call returned (= ms on the CPU). */
API Stats make_stats(const double *ms, const double *issue, int n, double t_start, double t_end) {
    Stats s;
    double sum = 0, isum = 0, var = 0;
    double *tmp;
    if (n < 1) die("no timed runs");
    tmp = dalloc(n);
    s.min = ms[0];
    for (int i = 0; i < n; i++) {
        sum += ms[i];
        isum += issue[i];
        if (ms[i] < s.min) s.min = ms[i];
        tmp[i] = ms[i];
    }
    s.mean = sum / n;
    s.issue = isum / n;
    for (int i = 0; i < n; i++) var += (ms[i] - s.mean) * (ms[i] - s.mean);
    s.std = n > 1 ? sqrt(var / (n - 1)) : 0.0;
    qsort(tmp, (size_t)n, sizeof(double), cmp_double);
    s.median = (n % 2) ? tmp[n / 2] : 0.5 * (tmp[n / 2 - 1] + tmp[n / 2]);
    free(tmp);
    s.t_start = t_start;
    s.t_end = t_end;
    return s;
}

/* ------------------------------------------------------------------ files */
API void mkdir_p(const char *path) {
    char buf[1024];
    size_t n = strlen(path);
    if (n >= sizeof buf) die("path too long: %s", path);
    memcpy(buf, path, n + 1);
    for (size_t i = 1; i <= n; i++) {
        if (buf[i] == '/' || buf[i] == '\\' || buf[i] == '\0') {
            char c = buf[i];
            buf[i] = '\0';
#ifdef _WIN32
            _mkdir(buf);
#else
            mkdir(buf, 0755);
#endif
            buf[i] = c;
        }
    }
}

API FILE *open_in(const char *dir, const char *name, const char *mode) {
    char path[1200];
    FILE *f;
    snprintf(path, sizeof path, "%s/%s", dir, name);
    f = fopen(path, mode);
    if (!f) die("cannot open %s", path);
    return f;
}

/* Same columns as the PyTorch version (code/common.py), so the same analysis scripts can read both. */
#define CSV_HEADER "exp,impl,device,threads,size,mean_ms,std_ms,min_ms,median_ms,issue_ms,warmup,repeats," \
                   "gflops,gbps,order,t_start,t_end,machine,timestamp"

API FILE *csv_open(const char *dir, const char *name) {
    char path[1200];
    int fresh = 1;
    FILE *probe, *f;
    snprintf(path, sizeof path, "%s/%s", dir, name);
    probe = fopen(path, "r");
    if (probe) {
        fresh = fgetc(probe) == EOF;
        fclose(probe);
    }
    f = fopen(path, "a");
    if (!f) die("cannot open %s", path);
    if (fresh) fprintf(f, "%s\n", CSV_HEADER);
    return f;
}

static int g_order = 0;

API void csv_row(FILE *f, const char *exp, const char *impl, const char *device, int threads, long size,
                 const Stats *s, int warmup, int repeat, double flops, double bytes, const char *machine) {
    char ts[40];
    double sec = s->mean / 1000.0;
    iso_utc(ts, sizeof ts);
    fprintf(f, "%s,%s,%s,%d,%ld,%.6f,%.6f,%.6f,%.6f,%.6f,%d,%d,", exp, impl, device, threads, size, s->mean,
            s->std, s->min, s->median, s->issue, warmup, repeat);
    if (flops > 0) fprintf(f, "%.3f", flops / sec / 1e9);
    fputc(',', f);
    if (bytes > 0) fprintf(f, "%.3f", bytes / sec / 1e9);
    fprintf(f, ",%d,%.4f,%.4f,%s,%s\n", ++g_order, s->t_start, s->t_end, machine, ts);
    fflush(f);
}

API void json_str(FILE *f, const char *key, const char *val, int comma) {
    fprintf(f, "  \"%s\": \"", key);
    for (const char *p = val; *p; p++) {
        if (*p == '"' || *p == '\\') fputc('\\', f);
        if ((unsigned char)*p >= 32) fputc(*p, f);
    }
    fprintf(f, "\"%s\n", comma ? "," : "");
}

API void cpu_model(char *buf, size_t n) {
    snprintf(buf, n, "unknown");
#ifndef _WIN32
    {
        FILE *f = fopen("/proc/cpuinfo", "r");
        char line[512];
        if (!f) return;
        while (fgets(line, sizeof line, f)) {
            if (!strncmp(line, "model name", 10)) {
                char *c = strchr(line, ':');
                if (c) {
                    c++;
                    while (*c == ' ' || *c == '\t') c++;
                    c[strcspn(c, "\n")] = 0;
                    snprintf(buf, n, "%s", c);
                }
                break;
            }
        }
        fclose(f);
    }
#else
    {
        const char *p = getenv("PROCESSOR_IDENTIFIER");
        if (p) snprintf(buf, n, "%s", p);
    }
#endif
}

/* Number of threads of this process (Linux /proc/self/status), -1 if unknown. */
API int process_threads(void) {
    int t = -1;
#ifndef _WIN32
    FILE *f = fopen("/proc/self/status", "r");
    char line[256];
    if (!f) return -1;
    while (fgets(line, sizeof line, f))
        if (!strncmp(line, "Threads:", 8)) t = atoi(line + 8);
    fclose(f);
#endif
    return t;
}

/* ------------------------------------------------------------------ command line */
typedef struct {
    const char *machine, *outdir, *exps;
    int warmup, repeat, order_seed, quick, check_only, no_check;
    double warm_seconds;
    const char *batches, *direct_batches, *mv_sizes, *gemm_sizes, *gemm_naive_sizes;
    const char *bw_sizes, *fc_sizes, *fc_b, *step_batches;
} Args;

#define ALL_EXPS "conv,fc,gemm,bw,fcb,step"

API void apply_quick(Args *a) {
    a->batches = "1,3";
    a->direct_batches = "1,2";
    a->mv_sizes = "256,1000";
    a->gemm_sizes = "128,200";
    a->gemm_naive_sizes = "128";
    a->bw_sizes = "256,1000";
    a->fc_sizes = "256,300";
    a->fc_b = "1,8";
    a->step_batches = "2";
    a->warmup = 1;
    a->repeat = 2;
    a->warm_seconds = 0.5;
}

API int parse_list(const char *s, long *out, int max) {
    int n = 0;
    const char *p = s;
    while (*p) {
        char *end;
        long v = strtol(p, &end, 10);
        if (end == p || v <= 0) die("bad list '%s' (comma separated positive integers)", s);
        if (n == max) die("list too long: %s", s);
        out[n++] = v;
        p = end;
        while (*p == ',' || *p == ' ') p++;
    }
    return n;
}

API int has_token(const char *list, const char *tok) {
    size_t len_tok = strlen(tok);
    const char *p = list;
    while (*p) {
        const char *e = strchr(p, ',');
        size_t len = e ? (size_t)(e - p) : strlen(p);
        if (len == len_tok && !strncmp(p, tok, len)) return 1;
        if (!e) break;
        p = e + 1;
    }
    return 0;
}

API void parse_args(int argc, char **argv, Args *a, const char *usage) {
    /* --quick first sets the tiny grid; options given explicitly then override it, in any order */
    for (int i = 1; i < argc; i++)
        if (!strcmp(argv[i], "--quick")) apply_quick(a);
    for (int i = 1; i < argc; i++) {
        const char *k = argv[i];
        const char *v = (i + 1 < argc) ? argv[i + 1] : NULL;
        int takes = 1;
        if (!strcmp(k, "--help") || !strcmp(k, "-h")) {
            printf("%s", usage);
            exit(0);
        } else if (!strcmp(k, "--quick")) {
            a->quick = 1, takes = 0;
        } else if (!strcmp(k, "--check-only")) {
            a->check_only = 1, takes = 0;
        } else if (!strcmp(k, "--no-check")) {
            a->no_check = 1, takes = 0;
        } else if (!v) {
            die("missing value for %s", k);
        } else if (!strcmp(k, "--machine")) a->machine = v;
        else if (!strcmp(k, "--outdir")) a->outdir = v;
        else if (!strcmp(k, "--exps")) a->exps = v;
        else if (!strcmp(k, "--warmup")) a->warmup = atoi(v);
        else if (!strcmp(k, "--repeat")) a->repeat = atoi(v);
        else if (!strcmp(k, "--order-seed")) a->order_seed = atoi(v);
        else if (!strcmp(k, "--warm-seconds")) a->warm_seconds = atof(v);
        else if (!strcmp(k, "--batches")) a->batches = v;
        else if (!strcmp(k, "--direct-batches")) a->direct_batches = v;
        else if (!strcmp(k, "--matvec-sizes")) a->mv_sizes = v;
        else if (!strcmp(k, "--gemm-sizes")) a->gemm_sizes = v;
        else if (!strcmp(k, "--gemm-naive-sizes")) a->gemm_naive_sizes = v;
        else if (!strcmp(k, "--bw-sizes")) a->bw_sizes = v;
        else if (!strcmp(k, "--fc-sizes")) a->fc_sizes = v;
        else if (!strcmp(k, "--fc-b")) a->fc_b = v;
        else if (!strcmp(k, "--step-batches")) a->step_batches = v;
        else die("unknown option %s (try --help)", k);
        i += takes;
    }
    if (a->repeat < 1 || a->warmup < 0) die("--repeat must be >= 1 and --warmup >= 0");
    if (!a->check_only && (!a->machine || !a->outdir)) die("--machine and --outdir are required (try --help)");
}

/* ------------------------------------------------------------------ float64 references for --check */
static int g_fail = 0;

/* max |got - ref| / max |ref|: 0 for identical results, about 1 for an indexing bug. NaN counts as a failure. */
API double rel_err(const float *got, const double *ref, long n) {
    double maxref = 0, maxerr = 0;
    for (long i = 0; i < n; i++) {
        double r = fabs(ref[i]), e = fabs((double)got[i] - ref[i]);
        if (r > maxref) maxref = r;
        if (e != e) e = INFINITY;
        if (e > maxerr) maxerr = e;
    }
    return maxerr / (maxref > 0 ? maxref : 1.0);
}

API void check(const char *what, double err, double tol) {
    int ok = err <= tol;
    say("check %-44s rel. error %.2e  %s", what, err, ok ? "ok" : "FAIL");
    if (!ok) g_fail++;
}

/* C = A B (+ bias per row of C); row-major, A is M x K, B is K x N. */
API void ref_gemm(int M, int N, int K, const float *A, const float *B, const float *bias, double *C) {
    for (int i = 0; i < M; i++)
        for (int j = 0; j < N; j++) C[(long)i * N + j] = bias ? bias[i] : 0.0;
    for (int i = 0; i < M; i++)
        for (int k = 0; k < K; k++) {
            double a = A[(long)i * K + k];
            for (int j = 0; j < N; j++) C[(long)i * N + j] += a * B[(long)k * N + j];
        }
}

/* out[b][oc][oh*54 + ow] for nb images, straight from the definition of the convolution. */
API void ref_conv(int nb, const float *x, const float *w, const float *bias, double *out) {
    for (int b = 0; b < nb; b++)
        for (int oc = 0; oc < C_OUT; oc++)
            for (int oh = 0; oh < OUT_HW; oh++)
                for (int ow = 0; ow < OUT_HW; ow++) {
                    double s = bias[oc];
                    for (int c = 0; c < C_IN; c++)
                        for (int kh = 0; kh < KS; kh++)
                            for (int kw = 0; kw < KS; kw++)
                                s += (double)w[((oc * C_IN + c) * KS + kh) * KS + kw] *
                                     x[b * IMG_ELEMS + (long)c * IMG * IMG + (long)(oh * STRIDE + kh) * IMG +
                                       ow * STRIDE + kw];
                    out[(long)b * OUT_ELEMS + (long)oc * PATCHES + oh * OUT_HW + ow] = s;
                }
}

API void ref_matvec(int n, const float *A, const float *x, double *y) {
    for (int i = 0; i < n; i++) {
        double s = 0;
        for (int k = 0; k < n; k++) s += (double)A[(long)i * n + k] * x[k];
        y[i] = s;
    }
}

/* Caffe's im2col for one image: col[r][p], r = (c*11 + kh)*11 + kw, p = oh*54 + ow.
   Used by the sequential version and as the reference for the GPU im2col kernel. */
API NOINLINE void im2col_cpu(const float *RESTRICT x, float *RESTRICT col) {
    for (int c = 0; c < C_IN; c++)
        for (int kh = 0; kh < KS; kh++)
            for (int kw = 0; kw < KS; kw++) {
                float *RESTRICT dst = col + (long)((c * KS + kh) * KS + kw) * PATCHES;
                for (int oh = 0; oh < OUT_HW; oh++) {
                    const float *src = x + (long)c * IMG * IMG + (long)(oh * STRIDE + kh) * IMG + kw;
                    for (int ow = 0; ow < OUT_HW; ow++) dst[oh * OUT_HW + ow] = src[ow * STRIDE];
                }
            }
}

#endif /* HPC_COMMON_H */

In [ ]:
%%writefile /kaggle/working/c_cuda/bench_seq.c
/* bench_seq.c - sequential C version of the benchmarks (1 thread, no BLAS, no OpenMP, no library).

   Same experiments and CSV columns as the PyTorch code in ../code, written by hand:
     conv  (paper Fig 1-2)  caffe   per image: im2col, then GEMM W(96x121) x col(121x2916) + bias  [main series]
                            direct  the convolution loops themselves, no im2col                    [extension]
     fc    (paper Fig 3-4)  matvec_naive  y = A x, one accumulator (a dependency chain)
                            matvec        y = A x, 32 independent accumulators                    [main series]
           (gemm)           gemm_naive (i,j,k loops), gemm_ikj (loop interchange),
                            gemm (cache + register blocking, packed B panel)                       [main series]
     bw    memory bandwidth: sum, dot, copy, triad (STREAM-like) and matvec, arrays of n*n floats
     fcb   FC layer with B input vectors, (n x n)(n x B): mm1 = matvec, mm16..mm128 = gemm
     step  caffe conv split into im2col only / GEMM only / both

   Build (Linux, Kaggle or WSL):  gcc -O3 -march=native -std=c11 -fno-tree-loop-distribute-patterns \
                                      -o bench_seq bench_seq.c -lm
   -fno-tree-loop-distribute-patterns keeps gcc from turning the copy loop into a call to memcpy.
   No -ffast-math: the compiler may vectorize only where the code itself has independent sums.

   Usage:  ./bench_seq --check-only
           ./bench_seq --machine kaggle --outdir results/kaggle/seq1 --order-seed 1 [--quick] */
#include "common.h"

static volatile float g_sink; /* results go here so that no computation can be dropped */

/* ================================================================== kernels */

/* ---- GEMM, row-major: C (M x N) = A (M x K) * B (K x N) + bias[i] (bias may be NULL) */

static NOINLINE void gemm_naive(int M, int N, int K, const float *RESTRICT A, const float *RESTRICT B,
                                const float *RESTRICT bias, float *RESTRICT C) {
    for (int i = 0; i < M; i++)
        for (int j = 0; j < N; j++) {
            float s = 0.f;
            for (int k = 0; k < K; k++) s += A[(long)i * K + k] * B[(long)k * N + j]; /* B read down a column */
            C[(long)i * N + j] = s + (bias ? bias[i] : 0.f);
        }
}

static NOINLINE void gemm_ikj(int M, int N, int K, const float *RESTRICT A, const float *RESTRICT B,
                              const float *RESTRICT bias, float *RESTRICT C) {
    for (int i = 0; i < M; i++) {
        float *RESTRICT c = C + (long)i * N;
        const float b0 = bias ? bias[i] : 0.f;
        for (int j = 0; j < N; j++) c[j] = b0;
        for (int k = 0; k < K; k++) {
            const float a = A[(long)i * K + k];
            const float *RESTRICT b = B + (long)k * N;
            for (int j = 0; j < N; j++) c[j] += a * b[j]; /* contiguous rows: vectorizable */
        }
    }
}

/* Blocked GEMM. For each block of KC values of k and each panel of NR = 16 columns, the panel of B is
   copied into a contiguous buffer (so it stays in L1 whatever N is), then every group of MR = 4 rows
   is computed by the micro-kernel, which keeps a 4 x 16 tile of C in registers over the k loop. */
enum { MR = 4, NR = 16, KC = 256 };

static inline void micro_4x16(int kc, int K, int N, const float *RESTRICT A, const float *RESTRICT Bp,
                              float *RESTRICT C) {
    float acc[MR][NR];
    for (int r = 0; r < MR; r++)
        for (int j = 0; j < NR; j++) acc[r][j] = C[(long)r * N + j];
    for (int k = 0; k < kc; k++) {
        const float *RESTRICT b = Bp + k * NR;
        const float a0 = A[k], a1 = A[K + k], a2 = A[2L * K + k], a3 = A[3L * K + k];
        for (int j = 0; j < NR; j++) {
            const float bj = b[j];
            acc[0][j] += a0 * bj;
            acc[1][j] += a1 * bj;
            acc[2][j] += a2 * bj;
            acc[3][j] += a3 * bj;
        }
    }
    for (int r = 0; r < MR; r++)
        for (int j = 0; j < NR; j++) C[(long)r * N + j] = acc[r][j];
}

static NOINLINE void gemm_blocked(int M, int N, int K, const float *RESTRICT A, const float *RESTRICT B,
                                  const float *RESTRICT bias, float *RESTRICT C) {
    static float Bp[KC * NR]; /* 16 KB packed panel of B */
    const int Mf = M - M % MR, Nf = N - N % NR;
    for (int i = 0; i < M; i++) {
        const float b0 = bias ? bias[i] : 0.f;
        float *RESTRICT c = C + (long)i * N;
        for (int j = 0; j < N; j++) c[j] = b0;
    }
    for (int k0 = 0; k0 < K; k0 += KC) {
        const int kc = (K - k0 < KC) ? K - k0 : KC;
        for (int j0 = 0; j0 < Nf; j0 += NR) {
            for (int k = 0; k < kc; k++)
                for (int j = 0; j < NR; j++) Bp[k * NR + j] = B[(long)(k0 + k) * N + j0 + j];
            for (int i0 = 0; i0 < Mf; i0 += MR)
                micro_4x16(kc, K, N, A + (long)i0 * K + k0, Bp, C + (long)i0 * N + j0);
        }
        /* edges: the last N % 16 columns of every row, and all columns of the last M % 4 rows */
        for (int i = 0; i < M; i++) {
            const int jstart = (i < Mf) ? Nf : 0;
            float *RESTRICT c = C + (long)i * N;
            if (jstart >= N) continue;
            for (int k = k0; k < k0 + kc; k++) {
                const float a = A[(long)i * K + k];
                const float *RESTRICT b = B + (long)k * N;
                for (int j = jstart; j < N; j++) c[j] += a * b[j];
            }
        }
    }
}

/* ---- convolution */

/* Caffe: one image at a time, im2col into one reused column buffer, then one GEMM with the bias. */
static NOINLINE void conv_caffe(int nb, const float *x, const float *w, const float *bias, float *col, float *out) {
    for (int b = 0; b < nb; b++) {
        im2col_cpu(x + b * IMG_ELEMS, col);
        gemm_blocked(C_OUT, PATCHES, PATCH_LEN, w, col, bias, out + b * OUT_ELEMS);
    }
}

/* Direct convolution, loops ordered so that the innermost loop runs along an output row. */
static NOINLINE void conv_direct(int nb, const float *RESTRICT x, const float *RESTRICT w,
                                 const float *RESTRICT bias, float *RESTRICT out) {
    for (int b = 0; b < nb; b++)
        for (int oc = 0; oc < C_OUT; oc++) {
            float *RESTRICT o = out + b * OUT_ELEMS + (long)oc * PATCHES;
            for (int p = 0; p < PATCHES; p++) o[p] = bias[oc];
            for (int c = 0; c < C_IN; c++)
                for (int kh = 0; kh < KS; kh++)
                    for (int kw = 0; kw < KS; kw++) {
                        const float wv = w[((oc * C_IN + c) * KS + kh) * KS + kw];
                        for (int oh = 0; oh < OUT_HW; oh++) {
                            const float *RESTRICT src =
                                x + b * IMG_ELEMS + (long)c * IMG * IMG + (long)(oh * STRIDE + kh) * IMG + kw;
                            float *RESTRICT orow = o + oh * OUT_HW;
                            for (int ow = 0; ow < OUT_HW; ow++) orow[ow] += wv * src[ow * STRIDE];
                        }
                    }
        }
}

/* ---- matrix-vector and STREAM-like kernels */

static NOINLINE void matvec_naive(int n, const float *RESTRICT A, const float *RESTRICT x, float *RESTRICT y) {
    for (int i = 0; i < n; i++) {
        float s = 0.f; /* one accumulator: each add waits for the previous one */
        for (int k = 0; k < n; k++) s += A[(long)i * n + k] * x[k];
        y[i] = s;
    }
}

enum { NACC = 32 }; /* independent partial sums: enough to hide the add latency and to vectorize */

static inline float dot32(const float *RESTRICT a, const float *RESTRICT b, long n) {
    float s[NACC] = {0}, t = 0.f;
    long k = 0;
    for (; k + NACC <= n; k += NACC)
        for (int u = 0; u < NACC; u++) s[u] += a[k + u] * b[k + u];
    for (; k < n; k++) t += a[k] * b[k];
    for (int u = 0; u < NACC; u++) t += s[u];
    return t;
}

static NOINLINE void matvec(int n, const float *RESTRICT A, const float *RESTRICT x, float *RESTRICT y) {
    for (int i = 0; i < n; i++) y[i] = dot32(A + (long)i * n, x, n);
}

static NOINLINE float sum_seq(const float *RESTRICT x, long n) {
    float s[NACC] = {0}, t = 0.f;
    long k = 0;
    for (; k + NACC <= n; k += NACC)
        for (int u = 0; u < NACC; u++) s[u] += x[k + u];
    for (; k < n; k++) t += x[k];
    for (int u = 0; u < NACC; u++) t += s[u];
    return t;
}

static NOINLINE float dot_seq(const float *RESTRICT x, const float *RESTRICT y, long n) { return dot32(x, y, n); }

static NOINLINE void copy_seq(const float *RESTRICT x, float *RESTRICT y, long n) {
    for (long i = 0; i < n; i++) y[i] = x[i];
}

#define TRIAD_S 3.0f
static NOINLINE void triad_seq(const float *RESTRICT b, const float *RESTRICT c, float *RESTRICT o, long n) {
    for (long i = 0; i < n; i++) o[i] = b[i] + TRIAD_S * c[i];
}

/* FC layer with B input vectors: Y (n x B) = A (n x n) X (n x B); B = 1 is the matvec. */
static void fc_batch(int n, int B, const float *A, const float *X, float *Y) {
    if (B == 1)
        matvec(n, A, X, Y);
    else
        gemm_blocked(n, B, n, A, X, NULL, Y);
}

/* ================================================================== timing */

typedef struct Job Job;
struct Job {
    void (*run)(Job *);
    int n, nb, B;
    long N;
    const float *x, *y, *w, *bias, *a;
    float *col, *out;
};

static void run_caffe(Job *j) { conv_caffe(j->nb, j->x, j->w, j->bias, j->col, j->out); }
static void run_direct(Job *j) { conv_direct(j->nb, j->x, j->w, j->bias, j->out); }
static void run_matvec_naive(Job *j) { matvec_naive(j->n, j->a, j->x, j->out); }
static void run_matvec(Job *j) { matvec(j->n, j->a, j->x, j->out); }
static void run_gemm_naive(Job *j) { gemm_naive(j->n, j->n, j->n, j->a, j->x, NULL, j->out); }
static void run_gemm_ikj(Job *j) { gemm_ikj(j->n, j->n, j->n, j->a, j->x, NULL, j->out); }
static void run_gemm(Job *j) { gemm_blocked(j->n, j->n, j->n, j->a, j->x, NULL, j->out); }
static void run_sum(Job *j) { g_sink = sum_seq(j->x, j->N); }
static void run_dot(Job *j) { g_sink = dot_seq(j->x, j->y, j->N); }
static void run_copy(Job *j) { copy_seq(j->x, j->out, j->N); }
static void run_triad(Job *j) { triad_seq(j->x, j->y, j->out, j->N); }
static void run_fcb(Job *j) { fc_batch(j->n, j->B, j->a, j->x, j->out); }
static void run_step_im2col(Job *j) {
    for (int b = 0; b < j->nb; b++) im2col_cpu(j->x + b * IMG_ELEMS, j->col);
}
static void run_step_gemm(Job *j) {
    for (int b = 0; b < j->nb; b++) gemm_blocked(C_OUT, PATCHES, PATCH_LEN, j->w, j->col, j->bias, j->out + b * OUT_ELEMS);
}

/* warmup untimed runs, then repeat timed runs (wall clock, ms). */
static Stats bench(Job *j, int warmup, int repeat) {
    double *ms = dalloc(repeat), t0, t1;
    Stats s;
    for (int i = 0; i < warmup; i++) j->run(j);
    t0 = epoch_s();
    for (int i = 0; i < repeat; i++) {
        double start = now_s();
        j->run(j);
        ms[i] = (now_s() - start) * 1e3;
    }
    t1 = epoch_s();
    s = make_stats(ms, ms, repeat, t0, t1);
    free(ms);
    return s;
}

static float *random_array(long n, unsigned long long seed) {
    float *p = falloc(n);
    rng_seed(seed);
    fill_random(p, n);
    return p;
}

static float *zero_array(long n) {
    float *p = falloc(n);
    fzero(p, n);
    return p;
}

static void keep(const float *p, long n) { g_sink = p[0] + p[n / 2] + p[n - 1]; }

/* ================================================================== experiments */

static void exp_conv(const Args *a) {
    static const char *impls[] = {"caffe", "direct"};
    FILE *csv = csv_open(a->outdir, "conv.csv");
    float *w = random_array((long)C_OUT * PATCH_LEN, 1), *bias = random_array(C_OUT, 2);
    float *col = zero_array(COL_ELEMS);
    for (int ii = 0; ii < 2; ii++) {
        long list[MAX_LIST];
        int cnt = parse_list(ii ? a->direct_batches : a->batches, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        for (int t = 0; t < cnt; t++) {
            Job j;
            Stats s;
            int nb = (int)list[t];
            memset(&j, 0, sizeof j);
            j.nb = nb, j.w = w, j.bias = bias, j.col = col;
            j.x = random_array(nb * IMG_ELEMS, 100 + nb);
            j.out = zero_array(nb * OUT_ELEMS);
            j.run = ii ? run_direct : run_caffe;
            s = bench(&j, a->warmup, a->repeat);
            keep(j.out, nb * OUT_ELEMS);
            csv_row(csv, "conv", impls[ii], "cpu", 1, nb, &s, a->warmup, a->repeat, FLOPS_PER_IMAGE * nb,
                    4.0 * nb * (IMG_ELEMS + OUT_ELEMS), a->machine);
            say("conv  cpu %-13s batch=%-5d %11.3f ms +- %.3f  (%.3f ms/img, %.1f GFLOP/s)", impls[ii], nb, s.mean,
                s.std, s.mean / nb, FLOPS_PER_IMAGE * nb / (s.mean * 1e6));
            ffree((float *)j.x);
            ffree(j.out);
        }
    }
    ffree(w), ffree(bias), ffree(col);
    fclose(csv);
}

static void exp_fc(const Args *a) {
    static const char *impls[] = {"matvec_naive", "matvec"};
    FILE *csv = csv_open(a->outdir, "fc.csv");
    for (int ii = 0; ii < 2; ii++) {
        long list[MAX_LIST];
        int cnt = parse_list(a->mv_sizes, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        for (int t = 0; t < cnt; t++) {
            Job j;
            Stats s;
            int n = (int)list[t];
            memset(&j, 0, sizeof j);
            j.n = n;
            j.a = random_array((long)n * n, 200 + n);
            j.x = random_array(n, 300 + n);
            j.out = zero_array(n);
            j.run = ii ? run_matvec : run_matvec_naive;
            s = bench(&j, a->warmup, a->repeat);
            keep(j.out, n);
            csv_row(csv, "fc", impls[ii], "cpu", 1, n, &s, a->warmup, a->repeat, 2.0 * n * n,
                    4.0 * ((double)n * n + 2.0 * n), a->machine);
            say("fc    cpu %-13s n=%-6d %11.3f ms +- %.3f  (%.2f GFLOP/s, %.2f GB/s)", impls[ii], n, s.mean, s.std,
                2.0 * n * n / (s.mean * 1e6), 4.0 * n * n / (s.mean * 1e6));
            ffree((float *)j.a), ffree((float *)j.x), ffree(j.out);
        }
    }
    fclose(csv);
}

static void exp_gemm(const Args *a) {
    static const char *impls[] = {"gemm_naive", "gemm_ikj", "gemm"};
    FILE *csv = csv_open(a->outdir, "fc.csv");
    for (int ii = 0; ii < 3; ii++) {
        long list[MAX_LIST];
        int cnt = parse_list(ii == 0 ? a->gemm_naive_sizes : a->gemm_sizes, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        for (int t = 0; t < cnt; t++) {
            Job j;
            Stats s;
            int n = (int)list[t];
            double flops = 2.0 * n * (double)n * n;
            memset(&j, 0, sizeof j);
            j.n = n;
            j.a = random_array((long)n * n, 400 + n);
            j.x = random_array((long)n * n, 500 + n);
            j.out = zero_array((long)n * n);
            j.run = ii == 0 ? run_gemm_naive : ii == 1 ? run_gemm_ikj : run_gemm;
            s = bench(&j, a->warmup, a->repeat);
            keep(j.out, (long)n * n);
            csv_row(csv, "fc", impls[ii], "cpu", 1, n, &s, a->warmup, a->repeat, flops, 12.0 * n * n, a->machine);
            say("gemm  cpu %-13s n=%-6d %11.3f ms +- %.3f  (%.2f GFLOP/s)", impls[ii], n, s.mean, s.std,
                flops / (s.mean * 1e6));
            ffree((float *)j.a), ffree((float *)j.x), ffree(j.out);
        }
    }
    fclose(csv);
}

static void exp_bw(const Args *a) {
    static const char *kernels[] = {"sum", "dot", "copy", "triad", "matvec"};
    static const double bytes_per[] = {4, 8, 8, 12, 4}, flops_per[] = {1, 2, 0, 2, 2};
    FILE *csv = csv_open(a->outdir, "bw.csv");
    for (int kk = 0; kk < 5; kk++) {
        long list[MAX_LIST];
        int cnt = parse_list(a->bw_sizes, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        for (int t = 0; t < cnt; t++) {
            Job j;
            Stats s;
            int n = (int)list[t];
            long N = (long)n * n;
            memset(&j, 0, sizeof j);
            j.n = n, j.N = N;
            if (kk == 4) { /* matvec over the same n*n floats */
                j.a = random_array(N, 600 + n);
                j.x = random_array(n, 700 + n);
                j.out = zero_array(n);
                j.run = run_matvec;
            } else {
                j.x = random_array(N, 600 + n);
                if (kk == 1 || kk == 3) j.y = random_array(N, 700 + n);
                if (kk >= 2) j.out = zero_array(N);
                j.run = kk == 0 ? run_sum : kk == 1 ? run_dot : kk == 2 ? run_copy : run_triad;
            }
            s = bench(&j, a->warmup, a->repeat);
            if (j.out) keep(j.out, kk == 4 ? n : N);
            csv_row(csv, "bw", kernels[kk], "cpu", 1, n, &s, a->warmup, a->repeat, flops_per[kk] * N,
                    bytes_per[kk] * N, a->machine);
            say("bw    cpu %-13s n=%-6d %11.3f ms +- %.3f  (%.2f GB/s)", kernels[kk], n, s.mean, s.std,
                bytes_per[kk] * N / (s.mean * 1e6));
            if (j.a) ffree((float *)j.a);
            if (j.x) ffree((float *)j.x);
            if (j.y) ffree((float *)j.y);
            if (j.out) ffree(j.out);
        }
    }
    fclose(csv);
}

static void exp_fcb(const Args *a) {
    long sizes[MAX_LIST], bs[MAX_LIST], order[MAX_LIST];
    int ns = parse_list(a->fc_sizes, sizes, MAX_LIST), nB = parse_list(a->fc_b, bs, MAX_LIST), cnt = 0;
    FILE *csv = csv_open(a->outdir, "fcb.csv");
    if (ns * nB > MAX_LIST) die("too many (n, B) pairs");
    for (int i = 0; i < ns * nB; i++) order[cnt++] = i;
    shuffle(order, cnt, a->order_seed);
    for (int t = 0; t < cnt; t++) {
        Job j;
        Stats s;
        char impl[32];
        int n = (int)sizes[order[t] / nB], B = (int)bs[order[t] % nB];
        double flops = 2.0 * n * (double)n * B;
        memset(&j, 0, sizeof j);
        j.n = n, j.B = B;
        j.a = random_array((long)n * n, 800 + n);
        j.x = random_array((long)n * B, 900 + B);
        j.out = zero_array((long)n * B);
        j.run = run_fcb;
        s = bench(&j, a->warmup, a->repeat);
        keep(j.out, (long)n * B);
        snprintf(impl, sizeof impl, "mm%d", B);
        csv_row(csv, "fcb", impl, "cpu", 1, n, &s, a->warmup, a->repeat, flops, 4.0 * n * n, a->machine);
        say("fcb   cpu B=%-4d n=%-6d %11.3f ms +- %.3f  (%.2f GFLOP/s)", B, n, s.mean, s.std, flops / (s.mean * 1e6));
        ffree((float *)j.a), ffree((float *)j.x), ffree(j.out);
    }
    fclose(csv);
}

static void exp_step(const Args *a) {
    static const char *steps[] = {"im2col", "gemm", "caffe"};
    FILE *csv = csv_open(a->outdir, "step.csv");
    float *w = random_array((long)C_OUT * PATCH_LEN, 1), *bias = random_array(C_OUT, 2);
    float *col = zero_array(COL_ELEMS);
    for (int ss = 0; ss < 3; ss++) {
        long list[MAX_LIST];
        int cnt = parse_list(a->step_batches, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        for (int t = 0; t < cnt; t++) {
            Job j;
            Stats s;
            int nb = (int)list[t];
            memset(&j, 0, sizeof j);
            j.nb = nb, j.w = w, j.bias = bias, j.col = col;
            j.x = random_array(nb * IMG_ELEMS, 100 + nb);
            j.out = zero_array(nb * OUT_ELEMS);
            im2col_cpu(j.x, col); /* GEMM-only needs valid columns */
            j.run = ss == 0 ? run_step_im2col : ss == 1 ? run_step_gemm : run_caffe;
            s = bench(&j, a->warmup, a->repeat);
            keep(j.out, nb * OUT_ELEMS);
            csv_row(csv, "step", steps[ss], "cpu", 1, nb, &s, a->warmup, a->repeat, ss ? FLOPS_PER_IMAGE * nb : 0, 0,
                    a->machine);
            say("step  cpu %-13s batch=%-5d %11.3f ms +- %.3f  (%.1f us/img)", steps[ss], nb, s.mean, s.std,
                s.mean / nb * 1e3);
            ffree((float *)j.x), ffree(j.out);
        }
    }
    ffree(w), ffree(bias), ffree(col);
    fclose(csv);
}

/* ================================================================== correctness */

static void check_gemm_shape(int M, int N, int K, int with_bias) {
    float *A = random_array((long)M * K, 11), *B = random_array((long)K * N, 12), *bias = random_array(M, 13);
    float *C = falloc((long)M * N);
    double *ref = dalloc((long)M * N);
    char what[96];
    const float *bb = with_bias ? bias : NULL;
    ref_gemm(M, N, K, A, B, bb, ref);
    gemm_naive(M, N, K, A, B, bb, C);
    snprintf(what, sizeof what, "gemm_naive   %dx%dx%d%s", M, N, K, with_bias ? " +bias" : "");
    check(what, rel_err(C, ref, (long)M * N), 1e-4);
    gemm_ikj(M, N, K, A, B, bb, C);
    snprintf(what, sizeof what, "gemm_ikj     %dx%dx%d%s", M, N, K, with_bias ? " +bias" : "");
    check(what, rel_err(C, ref, (long)M * N), 1e-4);
    gemm_blocked(M, N, K, A, B, bb, C);
    snprintf(what, sizeof what, "gemm         %dx%dx%d%s", M, N, K, with_bias ? " +bias" : "");
    check(what, rel_err(C, ref, (long)M * N), 1e-4);
    ffree(A), ffree(B), ffree(bias), ffree(C);
    free(ref);
}

static int run_checks(void) {
    g_fail = 0;
    /* GEMM: odd shapes (edges of the 4 x 16 tiles), more than one k block (K > 256), the conv shape */
    check_gemm_shape(37, 53, 29, 1);
    check_gemm_shape(5, 17, 3, 0);
    check_gemm_shape(64, 80, 300, 0);
    check_gemm_shape(C_OUT, PATCHES, PATCH_LEN, 1);
    { /* convolution, 2 images */
        const int nb = 2;
        float *x = random_array(nb * IMG_ELEMS, 21), *w = random_array((long)C_OUT * PATCH_LEN, 22);
        float *bias = random_array(C_OUT, 23), *col = falloc(COL_ELEMS), *out = falloc(nb * OUT_ELEMS);
        double *ref = dalloc(nb * OUT_ELEMS);
        ref_conv(nb, x, w, bias, ref);
        conv_caffe(nb, x, w, bias, col, out);
        check("conv caffe (im2col + gemm), 2 images", rel_err(out, ref, nb * OUT_ELEMS), 1e-4);
        conv_direct(nb, x, w, bias, out);
        check("conv direct, 2 images", rel_err(out, ref, nb * OUT_ELEMS), 1e-4);
        ffree(x), ffree(w), ffree(bias), ffree(col), ffree(out);
        free(ref);
    }
    { /* matvec */
        static const int sizes[] = {1, 37, 300, 1000};
        for (int t = 0; t < 4; t++) {
            int n = sizes[t];
            float *A = random_array((long)n * n, 31), *x = random_array(n, 32), *y = falloc(n);
            double *ref = dalloc(n);
            char what[64];
            ref_matvec(n, A, x, ref);
            matvec_naive(n, A, x, y);
            snprintf(what, sizeof what, "matvec_naive n=%d", n);
            check(what, rel_err(y, ref, n), 1e-4);
            matvec(n, A, x, y);
            snprintf(what, sizeof what, "matvec       n=%d", n);
            check(what, rel_err(y, ref, n), 1e-4);
            ffree(A), ffree(x), ffree(y);
            free(ref);
        }
    }
    { /* STREAM-like kernels on an odd length */
        const long N = 1000003;
        float *x = random_array(N, 41), *y = random_array(N, 42), *o = falloc(N);
        double s = 0, d = 0, sabs = 0, dabs = 0, got;
        double *ref = dalloc(N);
        for (long i = 0; i < N; i++) {
            s += x[i], d += (double)x[i] * y[i];
            sabs += fabs(x[i]), dabs += fabs((double)x[i] * y[i]);
        }
        /* error relative to the sum of magnitudes (the scale of float rounding in a long sum) */
        got = sum_seq(x, N);
        check("sum N=1000003", fabs(got - s) / sabs, 1e-6);
        got = dot_seq(x, y, N);
        check("dot N=1000003", fabs(got - d) / dabs, 1e-6);
        copy_seq(x, o, N);
        check("copy N=1000003 (exact)", memcmp(x, o, (size_t)N * sizeof(float)) ? 1.0 : 0.0, 0.0);
        triad_seq(x, y, o, N);
        for (long i = 0; i < N; i++) ref[i] = (double)x[i] + 3.0 * y[i];
        check("triad N=1000003", rel_err(o, ref, N), 1e-6);
        ffree(x), ffree(y), ffree(o);
        free(ref);
    }
    { /* FC with B vectors */
        const int n = 200, B = 8;
        float *A = random_array((long)n * n, 51), *X = random_array((long)n * B, 52), *Y = falloc((long)n * B);
        double *ref = dalloc((long)n * B);
        ref_gemm(n, B, n, A, X, NULL, ref);
        fc_batch(n, B, A, X, Y);
        check("fcb n=200 B=8", rel_err(Y, ref, (long)n * B), 1e-4);
        ref_gemm(n, 1, n, A, X, NULL, ref);
        fc_batch(n, 1, A, X, Y);
        check("fcb n=200 B=1 (matvec)", rel_err(Y, ref, n), 1e-4);
        ffree(A), ffree(X), ffree(Y);
        free(ref);
    }
    say("checks: %s (%d failed)", g_fail ? "FAILED" : "all passed", g_fail);
    return g_fail;
}

/* ================================================================== main */

static void write_env(const Args *a) {
    FILE *f = open_in(a->outdir, "env_seq.json", "w");
    char cpu[256], ts[40], buf[64];
    cpu_model(cpu, sizeof cpu);
    iso_utc(ts, sizeof ts);
    fprintf(f, "{\n");
    json_str(f, "program", "bench_seq (sequential C, 1 thread, no external library)", 1);
    json_str(f, "machine", a->machine, 1);
    json_str(f, "recorded_utc", ts, 1);
    json_str(f, "cpu", cpu, 1);
#ifdef __VERSION__
    json_str(f, "compiler", __VERSION__, 1);
#endif
    json_str(f, "experiments", a->exps, 1);
    json_str(f, "batches", a->batches, 1);
    json_str(f, "direct_batches", a->direct_batches, 1);
    json_str(f, "matvec_sizes", a->mv_sizes, 1);
    json_str(f, "gemm_sizes", a->gemm_sizes, 1);
    json_str(f, "gemm_naive_sizes", a->gemm_naive_sizes, 1);
    json_str(f, "bw_sizes", a->bw_sizes, 1);
    json_str(f, "fc_sizes", a->fc_sizes, 1);
    json_str(f, "fc_b", a->fc_b, 1);
    json_str(f, "step_batches", a->step_batches, 1);
    snprintf(buf, sizeof buf, "%d", process_threads());
    json_str(f, "process_threads", buf, 1);
    fprintf(f, "  \"warmup\": %d,\n  \"repeat\": %d,\n  \"order_seed\": %d,\n  \"quick\": %d\n}\n", a->warmup,
            a->repeat, a->order_seed, a->quick);
    fclose(f);
}

static const char *USAGE =
    "bench_seq - sequential C benchmarks (1 thread, no library)\n"
    "  --check-only            run the correctness checks and exit\n"
    "  --machine NAME          label written to every row (required unless --check-only)\n"
    "  --outdir DIR            run folder, created if needed; CSV files are appended (required)\n"
    "  --exps LIST             subset of " ALL_EXPS "\n"
    "  --warmup N --repeat N   untimed / timed runs per point (default 3 / 10)\n"
    "  --order-seed N          0 = ascending sizes, else shuffled (default 1)\n"
    "  --quick                 tiny grid, a few seconds\n"
    "  --no-check              skip the checks before timing\n"
    "  --batches, --direct-batches, --matvec-sizes, --gemm-sizes, --gemm-naive-sizes,\n"
    "  --bw-sizes, --fc-sizes, --fc-b, --step-batches LIST   comma separated grids\n";

int main(int argc, char **argv) {
    Args a;
    double start;
    memset(&a, 0, sizeof a);
    a.exps = ALL_EXPS;
    a.warmup = 3, a.repeat = 10, a.order_seed = 1;
    a.batches = "1,8,16,32,64,96,128,160,192,224,256";
    a.direct_batches = "1,16,64,256";
    a.mv_sizes = "256,512,1024,2048,4096,8192,12288,16384";
    a.gemm_sizes = "256,512,1024,1536,2048";
    a.gemm_naive_sizes = "256,512";
    a.bw_sizes = "1024,4096,8192,16384";
    a.fc_sizes = "2048,8192";
    a.fc_b = "1,16,64,128";
    a.step_batches = "16,64,256";
    parse_args(argc, argv, &a, USAGE);

    if (!a.no_check && run_checks()) return 1;
    if (a.check_only) return 0;

    mkdir_p(a.outdir);
    g_log = open_in(a.outdir, "log_seq.txt", "a");
    write_env(&a);
    say("bench_seq: machine %s, outdir %s, warmup %d, repeat %d, order seed %d%s", a.machine, a.outdir, a.warmup,
        a.repeat, a.order_seed, a.quick ? " (quick)" : "");
    start = now_s();
    if (has_token(a.exps, "conv")) exp_conv(&a);
    if (has_token(a.exps, "fc")) exp_fc(&a);
    if (has_token(a.exps, "gemm")) exp_gemm(&a);
    if (has_token(a.exps, "bw")) exp_bw(&a);
    if (has_token(a.exps, "fcb")) exp_fcb(&a);
    if (has_token(a.exps, "step")) exp_step(&a);
    say("threads in this process: %d (1 = sequential)", process_threads());
    say("done in %.1f s -> %s", now_s() - start, a.outdir);
    fclose(g_log);
    return 0;
}

In [ ]:
%%writefile /kaggle/working/c_cuda/bench_cuda.cu
// bench_cuda.cu - CUDA version of the benchmarks, every kernel written by hand (no cuBLAS, no cuDNN,
// no Thrust/CUB): the program links only the CUDA runtime.
//
// Same experiments and CSV columns as bench_seq.c and the PyTorch code in ../code:
//   conv  caffe    per image: im2col kernel, then GEMM kernel W(96x121) x col(121x2916) + bias  [main series]
//         batched  one im2col for the whole batch, one batched GEMM launch (grid.z = image)
//         direct   one thread per output value, weights in constant memory, no im2col
//   fc    matvec_naive  one thread per row (reads of a warp are n floats apart: not coalesced)
//         matvec        one warp per row, float4 coalesced reads, warp-shuffle reduction        [main series]
//         gemm_naive    one thread per element of C
//         gemm_tiled    16x16 tiles of A and B in shared memory
//         gemm          64-column tiles, each thread keeps a TM x 4 block of C in registers   [main series]
//   bw    sum, dot (two-pass reductions), copy, triad (float4 grid-stride loops), matvec
//   fcb   (n x n)(n x B): mm1 = matvec, mm16..mm128 = gemm
//   step  caffe conv split into im2col only / GEMM only / both
//
// Timing as in the PyTorch version: synchronize, start the host clock, launch, note the time when the
// launches return (issue_ms), synchronize, stop the clock. Data is already on the GPU; host<->device
// copies are not timed. Before each implementation the GPU is kept busy for --warm-seconds (GEMM).
//
// Build:  nvcc -O3 -std=c++17 -arch=sm_75 -o bench_cuda bench_cuda.cu      (sm_75 = Tesla T4)
// Usage:  ./bench_cuda --check-only
//         ./bench_cuda --machine kaggle --outdir results/kaggle/c_cuda1 --order-seed 1 --repeat 30
#include "common.h"

#include <cuda_runtime.h>

#define CUDA_CHECK(call)                                                                                   \
    do {                                                                                                   \
        cudaError_t e_ = (call);                                                                           \
        if (e_ != cudaSuccess)                                                                             \
            die("CUDA %s at %s:%d: %s", cudaGetErrorName(e_), __FILE__, __LINE__, cudaGetErrorString(e_)); \
    } while (0)
#define KERNEL_CHECK() CUDA_CHECK(cudaGetLastError())

static int g_sms = 40;          // multiprocessors
static int g_full_blocks = 160; // blocks of 256 threads that fill every SM once

static int grid_for(long n, int block) {
    long g = (n + block - 1) / block, cap = (long)g_sms * 32;
    return (int)(g < 1 ? 1 : g < cap ? g : cap);
}

// ===================================================================== kernels

__global__ void fill_k(float *__restrict__ p, long n, unsigned long long seed) {
    for (long i = blockIdx.x * (long)blockDim.x + threadIdx.x; i < n; i += (long)gridDim.x * blockDim.x) {
        unsigned long long z = seed * 0x9E3779B97F4A7C15ULL + (unsigned long long)i * 0xBF58476D1CE4E5B9ULL;
        z ^= z >> 31;
        z *= 0x94D049BB133111EBULL;
        z ^= z >> 29;
        p[i] = (float)(z >> 40) * (2.0f / 16777216.0f) - 1.0f;
    }
}

// ---- im2col for nb images: col[b][r][p], r = (c*11 + kh)*11 + kw, p = oh*54 + ow (Caffe layout).
// Consecutive threads write consecutive p, so the writes are coalesced.
__global__ void im2col_k(const float *__restrict__ x, float *__restrict__ col, int nb) {
    const long total = (long)nb * COL_ELEMS;
    for (long i = blockIdx.x * (long)blockDim.x + threadIdx.x; i < total; i += (long)gridDim.x * blockDim.x) {
        const int p = (int)(i % PATCHES);
        const long t = i / PATCHES;
        const int r = (int)(t % PATCH_LEN), b = (int)(t / PATCH_LEN);
        const int oh = p / OUT_HW, ow = p % OUT_HW;
        const int c = r / (KS * KS), kh = (r / KS) % KS, kw = r % KS;
        col[i] = x[b * IMG_ELEMS + (long)c * IMG * IMG + (long)(oh * STRIDE + kh) * IMG + ow * STRIDE + kw];
    }
}

// ---- GEMM, row-major: C (M x N) = A (M x K) * B (K x N) + bias[row]; blockIdx.z selects a matrix pair
// B + z*sB, C + z*sC (A is shared), which is how the batched conv runs every image in one launch.

__global__ void gemm_naive_k(int M, int N, int K, const float *__restrict__ A, const float *__restrict__ B,
                             const float *__restrict__ bias, float *__restrict__ C, long sB, long sC) {
    const int col = blockIdx.x * blockDim.x + threadIdx.x, row = blockIdx.y * blockDim.y + threadIdx.y;
    B += blockIdx.z * sB;
    C += blockIdx.z * sC;
    if (row >= M || col >= N) return;
    float s = 0.f;
    for (int k = 0; k < K; k++) s = fmaf(A[(long)row * K + k], B[(long)k * N + col], s);
    C[(long)row * N + col] = s + (bias ? bias[row] : 0.f);
}

template <int T>
__global__ void gemm_tiled_k(int M, int N, int K, const float *__restrict__ A, const float *__restrict__ B,
                             const float *__restrict__ bias, float *__restrict__ C, long sB, long sC) {
    __shared__ float As[T][T];
    __shared__ float Bs[T][T];
    const int tx = threadIdx.x, ty = threadIdx.y;
    const int row = blockIdx.y * T + ty, col = blockIdx.x * T + tx;
    float s = 0.f;
    B += blockIdx.z * sB;
    C += blockIdx.z * sC;
    for (int k0 = 0; k0 < K; k0 += T) {
        As[ty][tx] = (row < M && k0 + tx < K) ? A[(long)row * K + k0 + tx] : 0.f;
        Bs[ty][tx] = (k0 + ty < K && col < N) ? B[(long)(k0 + ty) * N + col] : 0.f;
        __syncthreads();
#pragma unroll
        for (int k = 0; k < T; k++) s = fmaf(As[ty][k], Bs[k][tx], s);
        __syncthreads();
    }
    if (row < M && col < N) C[(long)row * N + col] = s + (bias ? bias[row] : 0.f);
}

// Register blocking: a block of 16 x 16 threads computes a (16*TM) x 64 tile of C; thread (tx, ty) owns
// rows ty + 16i (i < TM) and columns tx + 16j (j < 4), so its 4*TM sums stay in registers. Tiles of
// A (stored transposed, padded) and B, 16 values of k deep, are staged in shared memory.
template <int TM>
__global__ void __launch_bounds__(256) gemm_reg_k(int M, int N, int K, const float *__restrict__ A,
                                                  const float *__restrict__ B, const float *__restrict__ bias,
                                                  float *__restrict__ C, long sB, long sC) {
    constexpr int BM = 16 * TM, BN = 64, BK = 16, TN = 4;
    __shared__ float As[BK][BM + 1];
    __shared__ float Bs[BK][BN];
    const int tx = threadIdx.x, ty = threadIdx.y, tid = ty * 16 + tx;
    const int m0 = blockIdx.y * BM, n0 = blockIdx.x * BN;
    float acc[TM][TN];
    B += blockIdx.z * sB;
    C += blockIdx.z * sC;
#pragma unroll
    for (int i = 0; i < TM; i++)
#pragma unroll
        for (int j = 0; j < TN; j++) acc[i][j] = 0.f;
    for (int k0 = 0; k0 < K; k0 += BK) {
        for (int e = tid; e < BM * BK; e += 256) { // consecutive threads: consecutive k of one row of A
            const int m = e / BK, k = e % BK, gm = m0 + m, gk = k0 + k;
            As[k][m] = (gm < M && gk < K) ? A[(long)gm * K + gk] : 0.f;
        }
        for (int e = tid; e < BK * BN; e += 256) { // consecutive threads: consecutive columns of B
            const int k = e / BN, n = e % BN, gk = k0 + k, gn = n0 + n;
            Bs[k][n] = (gk < K && gn < N) ? B[(long)gk * N + gn] : 0.f;
        }
        __syncthreads();
#pragma unroll
        for (int k = 0; k < BK; k++) {
            float a[TM], b[TN];
#pragma unroll
            for (int i = 0; i < TM; i++) a[i] = As[k][ty + 16 * i];
#pragma unroll
            for (int j = 0; j < TN; j++) b[j] = Bs[k][tx + 16 * j];
#pragma unroll
            for (int i = 0; i < TM; i++)
#pragma unroll
                for (int j = 0; j < TN; j++) acc[i][j] = fmaf(a[i], b[j], acc[i][j]);
        }
        __syncthreads();
    }
#pragma unroll
    for (int i = 0; i < TM; i++) {
        const int gm = m0 + ty + 16 * i;
        if (gm < M) {
            const float bv = bias ? bias[gm] : 0.f;
#pragma unroll
            for (int j = 0; j < TN; j++) {
                const int gn = n0 + tx + 16 * j;
                if (gn < N) C[(long)gm * N + gn] = acc[i][j] + bv;
            }
        }
    }
}

// ---- direct convolution: one thread per output value out[b][oc][p]. A warp shares one filter (p runs
// fastest), so the weights are read from constant memory as a broadcast.
__constant__ float c_w[C_OUT * PATCH_LEN];
__constant__ float c_bias[C_OUT];

__global__ void conv_direct_k(const float *__restrict__ x, float *__restrict__ out, int nb) {
    const long total = (long)nb * OUT_ELEMS;
    for (long i = blockIdx.x * (long)blockDim.x + threadIdx.x; i < total; i += (long)gridDim.x * blockDim.x) {
        const int p = (int)(i % PATCHES);
        const long t = i / PATCHES;
        const int oc = (int)(t % C_OUT), b = (int)(t / C_OUT);
        const int oh = p / OUT_HW, ow = p % OUT_HW;
        const float *src = x + b * IMG_ELEMS + (long)(oh * STRIDE) * IMG + ow * STRIDE;
        float s = c_bias[oc];
        for (int c = 0; c < C_IN; c++)
#pragma unroll
            for (int kh = 0; kh < KS; kh++)
#pragma unroll
                for (int kw = 0; kw < KS; kw++)
                    s = fmaf(c_w[oc * PATCH_LEN + (c * KS + kh) * KS + kw], src[(long)c * IMG * IMG + kh * IMG + kw], s);
        out[i] = s;
    }
}

// ---- matrix-vector
__global__ void matvec_naive_k(int n, const float *__restrict__ A, const float *__restrict__ x, float *__restrict__ y) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    const float *row = A + (long)i * n;
    float s = 0.f;
    for (int k = 0; k < n; k++) s = fmaf(row[k], x[k], s);
    y[i] = s;
}

__global__ void matvec_warp_k(int n, const float *__restrict__ A, const float *__restrict__ x, float *__restrict__ y) {
    const int row = (int)((blockIdx.x * (long)blockDim.x + threadIdx.x) >> 5), lane = threadIdx.x & 31;
    if (row >= n) return; // the whole warp leaves together
    const float *a = A + (long)row * n;
    float s = 0.f;
    if ((n & 3) == 0) { // rows start on a 16-byte boundary: read float4
        const float4 *a4 = reinterpret_cast<const float4 *>(a), *x4 = reinterpret_cast<const float4 *>(x);
        for (int k = lane; k < n / 4; k += 32) {
            const float4 u = a4[k], v = x4[k];
            s = fmaf(u.x, v.x, s);
            s = fmaf(u.y, v.y, s);
            s = fmaf(u.z, v.z, s);
            s = fmaf(u.w, v.w, s);
        }
    } else {
        for (int k = lane; k < n; k += 32) s = fmaf(a[k], x[k], s);
    }
    for (int off = 16; off > 0; off >>= 1) s += __shfl_down_sync(0xffffffffu, s, off);
    if (lane == 0) y[row] = s;
}

// ---- STREAM-like kernels
__device__ float block_reduce_sum(float v) { // result valid in thread 0
    __shared__ float warp_part[32];
    const int lane = threadIdx.x & 31, wid = threadIdx.x >> 5, nwarps = (blockDim.x + 31) >> 5;
    for (int off = 16; off > 0; off >>= 1) v += __shfl_down_sync(0xffffffffu, v, off);
    if (lane == 0) warp_part[wid] = v;
    __syncthreads();
    if (wid == 0) {
        v = lane < nwarps ? warp_part[lane] : 0.f;
        for (int off = 16; off > 0; off >>= 1) v += __shfl_down_sync(0xffffffffu, v, off);
    }
    return v;
}

__global__ void sum_partial_k(const float *__restrict__ x, long n, float *__restrict__ part) {
    const long tid = blockIdx.x * (long)blockDim.x + threadIdx.x, stride = (long)gridDim.x * blockDim.x;
    const float4 *x4 = reinterpret_cast<const float4 *>(x);
    const long n4 = n / 4;
    float s = 0.f;
    for (long i = tid; i < n4; i += stride) {
        const float4 v = x4[i];
        s += (v.x + v.y) + (v.z + v.w);
    }
    for (long i = 4 * n4 + tid; i < n; i += stride) s += x[i];
    s = block_reduce_sum(s);
    if (threadIdx.x == 0) part[blockIdx.x] = s;
}

__global__ void dot_partial_k(const float *__restrict__ x, const float *__restrict__ y, long n, float *__restrict__ part) {
    const long tid = blockIdx.x * (long)blockDim.x + threadIdx.x, stride = (long)gridDim.x * blockDim.x;
    const float4 *x4 = reinterpret_cast<const float4 *>(x), *y4 = reinterpret_cast<const float4 *>(y);
    const long n4 = n / 4;
    float s = 0.f;
    for (long i = tid; i < n4; i += stride) {
        const float4 u = x4[i], v = y4[i];
        s = fmaf(u.x, v.x, s);
        s = fmaf(u.y, v.y, s);
        s = fmaf(u.z, v.z, s);
        s = fmaf(u.w, v.w, s);
    }
    for (long i = 4 * n4 + tid; i < n; i += stride) s = fmaf(x[i], y[i], s);
    s = block_reduce_sum(s);
    if (threadIdx.x == 0) part[blockIdx.x] = s;
}

__global__ void final_sum_k(const float *__restrict__ part, int m, float *__restrict__ out) {
    float s = 0.f;
    for (int i = threadIdx.x; i < m; i += blockDim.x) s += part[i];
    s = block_reduce_sum(s);
    if (threadIdx.x == 0) out[0] = s;
}

__global__ void copy_k(const float *__restrict__ x, float *__restrict__ y, long n) {
    const long tid = blockIdx.x * (long)blockDim.x + threadIdx.x, stride = (long)gridDim.x * blockDim.x;
    const float4 *x4 = reinterpret_cast<const float4 *>(x);
    float4 *y4 = reinterpret_cast<float4 *>(y);
    for (long i = tid; i < n / 4; i += stride) y4[i] = x4[i];
    for (long i = 4 * (n / 4) + tid; i < n; i += stride) y[i] = x[i];
}

#define TRIAD_S 3.0f
__global__ void triad_k(const float *__restrict__ b, const float *__restrict__ c, float *__restrict__ o, long n) {
    const long tid = blockIdx.x * (long)blockDim.x + threadIdx.x, stride = (long)gridDim.x * blockDim.x;
    const float4 *b4 = reinterpret_cast<const float4 *>(b), *c4 = reinterpret_cast<const float4 *>(c);
    float4 *o4 = reinterpret_cast<float4 *>(o);
    for (long i = tid; i < n / 4; i += stride) {
        const float4 u = b4[i], v = c4[i];
        o4[i] = make_float4(u.x + TRIAD_S * v.x, u.y + TRIAD_S * v.y, u.z + TRIAD_S * v.z, u.w + TRIAD_S * v.w);
    }
    for (long i = 4 * (n / 4) + tid; i < n; i += stride) o[i] = b[i] + TRIAD_S * c[i];
}

// ===================================================================== launchers

enum GemmKind { G_NAIVE, G_TILED, G_REG };

static void gemm_gpu(GemmKind kind, int M, int N, int K, const float *A, const float *B, const float *bias, float *C,
                     int batch = 1, long sB = 0, long sC = 0) {
    if (kind == G_NAIVE) {
        dim3 blk(32, 8), grd((N + 31) / 32, (M + 7) / 8, batch);
        gemm_naive_k<<<grd, blk>>>(M, N, K, A, B, bias, C, sB, sC);
    } else if (kind == G_TILED) {
        dim3 blk(16, 16), grd((N + 15) / 16, (M + 15) / 16, batch);
        gemm_tiled_k<16><<<grd, blk>>>(M, N, K, A, B, bias, C, sB, sC);
    } else if (M % 64 != 0 && M % 48 == 0) { // M = 96 (the conv): two 48-row tiles, nothing wasted
        dim3 blk(16, 16), grd((N + 63) / 64, M / 48, batch);
        gemm_reg_k<3><<<grd, blk>>>(M, N, K, A, B, bias, C, sB, sC);
    } else {
        dim3 blk(16, 16), grd((N + 63) / 64, (M + 63) / 64, batch);
        gemm_reg_k<4><<<grd, blk>>>(M, N, K, A, B, bias, C, sB, sC);
    }
}

static void im2col_gpu(const float *x, float *col, int nb) { im2col_k<<<grid_for(nb * COL_ELEMS, 256), 256>>>(x, col, nb); }

static void conv_caffe_gpu(int nb, const float *x, const float *w, const float *bias, float *col, float *out) {
    for (int b = 0; b < nb; b++) { // two launches per image, as in Caffe
        im2col_gpu(x + b * IMG_ELEMS, col, 1);
        gemm_gpu(G_REG, C_OUT, PATCHES, PATCH_LEN, w, col, bias, out + b * OUT_ELEMS);
    }
}

static void conv_batched_gpu(int nb, const float *x, const float *w, const float *bias, float *cols, float *out) {
    im2col_gpu(x, cols, nb);
    gemm_gpu(G_REG, C_OUT, PATCHES, PATCH_LEN, w, cols, bias, out, nb, COL_ELEMS, OUT_ELEMS);
}

static void conv_direct_gpu(int nb, const float *x, float *out) {
    conv_direct_k<<<grid_for(nb * OUT_ELEMS, 256), 256>>>(x, out, nb);
}

static void set_direct_weights(const float *w_host, const float *bias_host) {
    CUDA_CHECK(cudaMemcpyToSymbol(c_w, w_host, sizeof(float) * C_OUT * PATCH_LEN));
    CUDA_CHECK(cudaMemcpyToSymbol(c_bias, bias_host, sizeof(float) * C_OUT));
}

static void matvec_gpu(int naive, int n, const float *A, const float *x, float *y) {
    if (naive)
        matvec_naive_k<<<(n + 255) / 256, 256>>>(n, A, x, y);
    else
        matvec_warp_k<<<(n + 7) / 8, 256>>>(n, A, x, y);
}

static void sum_gpu(const float *x, long n, float *part, float *out) {
    sum_partial_k<<<g_full_blocks, 256>>>(x, n, part);
    final_sum_k<<<1, 256>>>(part, g_full_blocks, out);
}

static void dot_gpu(const float *x, const float *y, long n, float *part, float *out) {
    dot_partial_k<<<g_full_blocks, 256>>>(x, y, n, part);
    final_sum_k<<<1, 256>>>(part, g_full_blocks, out);
}

static void fc_batch_gpu(int n, int B, const float *A, const float *X, float *Y) {
    if (B == 1)
        matvec_gpu(0, n, A, X, Y);
    else
        gemm_gpu(G_REG, n, B, n, A, X, NULL, Y);
}

// ===================================================================== memory helpers

static float *dmalloc(long n) {
    void *p = NULL;
    CUDA_CHECK(cudaMalloc(&p, (size_t)(n > 0 ? n : 1) * sizeof(float)));
    return (float *)p;
}
static float *drandom(long n, unsigned long long seed) {
    float *p = dmalloc(n);
    fill_k<<<g_full_blocks, 256>>>(p, n, seed);
    KERNEL_CHECK();
    return p;
}
static float *dzero(long n) {
    float *p = dmalloc(n);
    CUDA_CHECK(cudaMemset(p, 0, (size_t)n * sizeof(float)));
    return p;
}
static float *to_dev(const float *h, long n) {
    float *p = dmalloc(n);
    CUDA_CHECK(cudaMemcpy(p, h, (size_t)n * sizeof(float), cudaMemcpyHostToDevice));
    return p;
}
static void to_host(float *h, const float *d, long n) {
    CUDA_CHECK(cudaMemcpy(h, d, (size_t)n * sizeof(float), cudaMemcpyDeviceToHost));
}
static void dfree(const float *p) { CUDA_CHECK(cudaFree((void *)p)); }
static float *host_random(long n, unsigned long long seed) {
    float *p = falloc(n);
    rng_seed(seed);
    fill_random(p, n);
    return p;
}

// ===================================================================== timing

template <class F>
static Stats bench_gpu(F fn, int warmup, int repeat) {
    double *ms = dalloc(repeat), *iss = dalloc(repeat), t0, t1;
    Stats s;
    for (int i = 0; i < warmup; i++) fn();
    CUDA_CHECK(cudaDeviceSynchronize());
    t0 = epoch_s();
    for (int i = 0; i < repeat; i++) {
        CUDA_CHECK(cudaDeviceSynchronize());
        const double start = now_s();
        fn();
        iss[i] = (now_s() - start) * 1e3;
        CUDA_CHECK(cudaDeviceSynchronize());
        ms[i] = (now_s() - start) * 1e3;
    }
    t1 = epoch_s();
    KERNEL_CHECK();
    s = make_stats(ms, iss, repeat, t0, t1);
    free(ms), free(iss);
    return s;
}

// Keep the GPU busy so it reaches its sustained clocks before anything is timed (as gpu_warmup in ../code).
static void gpu_warmup(double seconds) {
    if (seconds <= 0) return;
    const int n = 2048;
    float *a = drandom((long)n * n, 1), *c = dzero((long)n * n);
    const double end = now_s() + seconds;
    while (now_s() < end) {
        gemm_gpu(G_REG, n, n, n, a, a, NULL, c);
        CUDA_CHECK(cudaDeviceSynchronize());
    }
    dfree(a), dfree(c);
}

static void row(FILE *csv, const char *exp, const char *impl, long size, const Stats &s, const Args *a, double flops,
                double bytes) {
    csv_row(csv, exp, impl, "cuda", 1, size, &s, a->warmup, a->repeat, flops, bytes, a->machine);
}

// ===================================================================== experiments

static void exp_conv(const Args *a) {
    static const char *impls[] = {"caffe", "batched", "direct"};
    FILE *csv = csv_open(a->outdir, "conv.csv");
    float *hw = host_random((long)C_OUT * PATCH_LEN, 1), *hb = host_random(C_OUT, 2);
    float *w = to_dev(hw, (long)C_OUT * PATCH_LEN), *bias = to_dev(hb, C_OUT);
    set_direct_weights(hw, hb);
    for (int ii = 0; ii < 3; ii++) {
        long list[MAX_LIST];
        int cnt = parse_list(ii == 2 ? a->direct_batches : a->batches, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        gpu_warmup(a->warm_seconds);
        for (int t = 0; t < cnt; t++) {
            const int nb = (int)list[t];
            float *x = drandom(nb * IMG_ELEMS, 100 + nb), *out = dzero(nb * OUT_ELEMS);
            float *col = dzero((ii == 1 ? nb : 1) * COL_ELEMS);
            Stats s;
            if (ii == 0)
                s = bench_gpu([&] { conv_caffe_gpu(nb, x, w, bias, col, out); }, a->warmup, a->repeat);
            else if (ii == 1)
                s = bench_gpu([&] { conv_batched_gpu(nb, x, w, bias, col, out); }, a->warmup, a->repeat);
            else
                s = bench_gpu([&] { conv_direct_gpu(nb, x, out); }, a->warmup, a->repeat);
            row(csv, "conv", impls[ii], nb, s, a, FLOPS_PER_IMAGE * nb, 4.0 * nb * (IMG_ELEMS + OUT_ELEMS));
            say("conv  cuda %-13s batch=%-5d %10.3f ms +- %.3f  (%.1f us/img, %.0f GFLOP/s)  issue %.3f ms", impls[ii],
                nb, s.mean, s.std, s.mean / nb * 1e3, FLOPS_PER_IMAGE * nb / (s.mean * 1e6), s.issue);
            dfree(x), dfree(out), dfree(col);
        }
    }
    dfree(w), dfree(bias);
    ffree(hw), ffree(hb);
    fclose(csv);
}

static void exp_fc(const Args *a) {
    static const char *impls[] = {"matvec_naive", "matvec"};
    FILE *csv = csv_open(a->outdir, "fc.csv");
    for (int ii = 0; ii < 2; ii++) {
        long list[MAX_LIST];
        int cnt = parse_list(a->mv_sizes, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        gpu_warmup(a->warm_seconds);
        for (int t = 0; t < cnt; t++) {
            const int n = (int)list[t];
            float *A = drandom((long)n * n, 200 + n), *x = drandom(n, 300 + n), *y = dzero(n);
            const int naive = ii == 0;
            Stats s = bench_gpu([&] { matvec_gpu(naive, n, A, x, y); }, a->warmup, a->repeat);
            row(csv, "fc", impls[ii], n, s, a, 2.0 * n * n, 4.0 * ((double)n * n + 2.0 * n));
            say("fc    cuda %-13s n=%-6d %10.3f ms +- %.3f  (%.1f GFLOP/s, %.1f GB/s)", impls[ii], n, s.mean, s.std,
                2.0 * n * n / (s.mean * 1e6), 4.0 * n * n / (s.mean * 1e6));
            dfree(A), dfree(x), dfree(y);
        }
    }
    fclose(csv);
}

static void exp_gemm(const Args *a) {
    static const char *impls[] = {"gemm_naive", "gemm_tiled", "gemm"};
    static const GemmKind kinds[] = {G_NAIVE, G_TILED, G_REG};
    FILE *csv = csv_open(a->outdir, "fc.csv");
    for (int ii = 0; ii < 3; ii++) {
        long list[MAX_LIST];
        int cnt = parse_list(ii == 0 ? a->gemm_naive_sizes : a->gemm_sizes, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        gpu_warmup(a->warm_seconds);
        for (int t = 0; t < cnt; t++) {
            const int n = (int)list[t];
            const double flops = 2.0 * n * (double)n * n;
            float *A = drandom((long)n * n, 400 + n), *B = drandom((long)n * n, 500 + n), *C = dzero((long)n * n);
            const GemmKind kind = kinds[ii];
            Stats s = bench_gpu([&] { gemm_gpu(kind, n, n, n, A, B, NULL, C); }, a->warmup, a->repeat);
            row(csv, "fc", impls[ii], n, s, a, flops, 12.0 * n * n);
            say("gemm  cuda %-13s n=%-6d %10.3f ms +- %.3f  (%.0f GFLOP/s)", impls[ii], n, s.mean, s.std,
                flops / (s.mean * 1e6));
            dfree(A), dfree(B), dfree(C);
        }
    }
    fclose(csv);
}

static void exp_bw(const Args *a) {
    static const char *kernels[] = {"sum", "dot", "copy", "triad", "matvec"};
    static const double bytes_per[] = {4, 8, 8, 12, 4}, flops_per[] = {1, 2, 0, 2, 2};
    FILE *csv = csv_open(a->outdir, "bw.csv");
    float *part = dzero(g_full_blocks), *res = dzero(1);
    for (int kk = 0; kk < 5; kk++) {
        long list[MAX_LIST];
        int cnt = parse_list(a->bw_sizes, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        gpu_warmup(a->warm_seconds);
        for (int t = 0; t < cnt; t++) {
            const int n = (int)list[t];
            const long N = (long)n * n;
            float *x = drandom(N, 600 + n);
            float *y = (kk == 1 || kk == 3) ? drandom(N, 700 + n) : kk == 4 ? drandom(n, 700 + n) : NULL;
            float *o = (kk == 2 || kk == 3) ? dzero(N) : kk == 4 ? dzero(n) : NULL;
            Stats s;
            if (kk == 0)
                s = bench_gpu([&] { sum_gpu(x, N, part, res); }, a->warmup, a->repeat);
            else if (kk == 1)
                s = bench_gpu([&] { dot_gpu(x, y, N, part, res); }, a->warmup, a->repeat);
            else if (kk == 2)
                s = bench_gpu([&] { copy_k<<<g_full_blocks, 256>>>(x, o, N); }, a->warmup, a->repeat);
            else if (kk == 3)
                s = bench_gpu([&] { triad_k<<<g_full_blocks, 256>>>(x, y, o, N); }, a->warmup, a->repeat);
            else
                s = bench_gpu([&] { matvec_gpu(0, n, x, y, o); }, a->warmup, a->repeat);
            row(csv, "bw", kernels[kk], n, s, a, flops_per[kk] * N, bytes_per[kk] * N);
            say("bw    cuda %-13s n=%-6d %10.3f ms +- %.3f  (%.1f GB/s)", kernels[kk], n, s.mean, s.std,
                bytes_per[kk] * N / (s.mean * 1e6));
            dfree(x);
            if (y) dfree(y);
            if (o) dfree(o);
        }
    }
    dfree(part), dfree(res);
    fclose(csv);
}

static void exp_fcb(const Args *a) {
    long sizes[MAX_LIST], bs[MAX_LIST], order[MAX_LIST];
    const int ns = parse_list(a->fc_sizes, sizes, MAX_LIST), nB = parse_list(a->fc_b, bs, MAX_LIST);
    int cnt = 0;
    FILE *csv = csv_open(a->outdir, "fcb.csv");
    if (ns * nB > MAX_LIST) die("too many (n, B) pairs");
    for (int i = 0; i < ns * nB; i++) order[cnt++] = i;
    shuffle(order, cnt, a->order_seed);
    gpu_warmup(a->warm_seconds);
    for (int t = 0; t < cnt; t++) {
        const int n = (int)sizes[order[t] / nB], B = (int)bs[order[t] % nB];
        const double flops = 2.0 * n * (double)n * B;
        float *A = drandom((long)n * n, 800 + n), *X = drandom((long)n * B, 900 + B), *Y = dzero((long)n * B);
        char impl[32];
        Stats s = bench_gpu([&] { fc_batch_gpu(n, B, A, X, Y); }, a->warmup, a->repeat);
        snprintf(impl, sizeof impl, "mm%d", B);
        row(csv, "fcb", impl, n, s, a, flops, 4.0 * n * n);
        say("fcb   cuda B=%-4d n=%-6d %10.3f ms +- %.3f  (%.0f GFLOP/s)", B, n, s.mean, s.std, flops / (s.mean * 1e6));
        dfree(A), dfree(X), dfree(Y);
    }
    fclose(csv);
}

static void exp_step(const Args *a) {
    static const char *steps[] = {"im2col", "gemm", "caffe"};
    FILE *csv = csv_open(a->outdir, "step.csv");
    float *w = drandom((long)C_OUT * PATCH_LEN, 1), *bias = drandom(C_OUT, 2), *col = dzero(COL_ELEMS);
    for (int ss = 0; ss < 3; ss++) {
        long list[MAX_LIST];
        int cnt = parse_list(a->step_batches, list, MAX_LIST);
        shuffle(list, cnt, a->order_seed);
        gpu_warmup(a->warm_seconds);
        for (int t = 0; t < cnt; t++) {
            const int nb = (int)list[t];
            float *x = drandom(nb * IMG_ELEMS, 100 + nb), *out = dzero(nb * OUT_ELEMS);
            Stats s;
            im2col_gpu(x, col, 1); // GEMM-only needs valid columns
            KERNEL_CHECK();
            if (ss == 0)
                s = bench_gpu([&] { for (int b = 0; b < nb; b++) im2col_gpu(x + b * IMG_ELEMS, col, 1); },
                              a->warmup, a->repeat);
            else if (ss == 1)
                s = bench_gpu([&] {
                        for (int b = 0; b < nb; b++)
                            gemm_gpu(G_REG, C_OUT, PATCHES, PATCH_LEN, w, col, bias, out + b * OUT_ELEMS);
                    }, a->warmup, a->repeat);
            else
                s = bench_gpu([&] { conv_caffe_gpu(nb, x, w, bias, col, out); }, a->warmup, a->repeat);
            row(csv, "step", steps[ss], nb, s, a, ss ? FLOPS_PER_IMAGE * nb : 0, 0);
            say("step  cuda %-13s batch=%-5d %10.3f ms +- %.3f  (%.1f us/img)  issue %.3f ms", steps[ss], nb, s.mean,
                s.std, s.mean / nb * 1e3, s.issue);
            dfree(x), dfree(out);
        }
    }
    dfree(w), dfree(bias), dfree(col);
    fclose(csv);
}

// ===================================================================== correctness (against float64 on the CPU)

static double gpu_vs_ref(const float *d, const double *ref, long n) {
    float *h = falloc(n);
    double e;
    CUDA_CHECK(cudaDeviceSynchronize());
    to_host(h, d, n);
    e = rel_err(h, ref, n);
    ffree(h);
    return e;
}

static void check_gemm_shape(int M, int N, int K, int with_bias, int batch) {
    static const char *names[] = {"gemm_naive", "gemm_tiled", "gemm"};
    const long sB = (long)K * N, sC = (long)M * N;
    float *hA = host_random((long)M * K, 11), *hB = host_random(sB * batch, 12), *hbias = host_random(M, 13);
    double *ref = dalloc(sC * batch);
    float *A = to_dev(hA, (long)M * K), *B = to_dev(hB, sB * batch), *bias = to_dev(hbias, M), *C = dzero(sC * batch);
    for (int z = 0; z < batch; z++) ref_gemm(M, N, K, hA, hB + z * sB, with_bias ? hbias : NULL, ref + z * sC);
    for (int kind = 0; kind < 3; kind++) {
        char what[96];
        CUDA_CHECK(cudaMemset(C, 0, (size_t)(sC * batch) * sizeof(float)));
        gemm_gpu((GemmKind)kind, M, N, K, A, B, with_bias ? bias : NULL, C, batch, sB, sC);
        KERNEL_CHECK();
        snprintf(what, sizeof what, "%-11s %dx%dx%d%s%s", names[kind], M, N, K, with_bias ? " +bias" : "",
                 batch > 1 ? " batch 3" : "");
        check(what, gpu_vs_ref(C, ref, sC * batch), 1e-4);
    }
    dfree(A), dfree(B), dfree(bias), dfree(C);
    ffree(hA), ffree(hB), ffree(hbias);
    free(ref);
}

static int run_checks(void) {
    g_fail = 0;
    // GEMM: odd shapes (tile edges), K > tile depth, both register-tile shapes (M = 96 uses 48-row tiles), batch
    check_gemm_shape(37, 53, 29, 1, 1);
    check_gemm_shape(5, 17, 3, 0, 1);
    check_gemm_shape(64, 80, 300, 0, 1);
    check_gemm_shape(48, 100, 20, 1, 1);
    check_gemm_shape(200, 8, 200, 0, 1);
    check_gemm_shape(C_OUT, PATCHES, PATCH_LEN, 1, 3);
    { // im2col and the three convolutions, 3 images
        const int nb = 3;
        float *hx = host_random(nb * IMG_ELEMS, 21), *hw = host_random((long)C_OUT * PATCH_LEN, 22);
        float *hb = host_random(C_OUT, 23), *hcol = falloc(COL_ELEMS), *gcol = falloc(nb * COL_ELEMS);
        double *ref = dalloc(nb * OUT_ELEMS);
        float *x = to_dev(hx, nb * IMG_ELEMS), *w = to_dev(hw, (long)C_OUT * PATCH_LEN), *bias = to_dev(hb, C_OUT);
        float *cols = dzero(nb * COL_ELEMS), *out = dzero(nb * OUT_ELEMS);
        int same = 1;
        im2col_gpu(x, cols, nb);
        KERNEL_CHECK();
        to_host(gcol, cols, nb * COL_ELEMS);
        for (int b = 0; b < nb; b++) {
            im2col_cpu(hx + b * IMG_ELEMS, hcol);
            same &= !memcmp(hcol, gcol + b * COL_ELEMS, sizeof(float) * COL_ELEMS);
        }
        check("im2col kernel = im2col on the CPU (exact)", same ? 0.0 : 1.0, 0.0);
        ref_conv(nb, hx, hw, hb, ref);
        set_direct_weights(hw, hb);
        conv_caffe_gpu(nb, x, w, bias, cols, out);
        KERNEL_CHECK();
        check("conv caffe, 3 images", gpu_vs_ref(out, ref, nb * OUT_ELEMS), 1e-4);
        CUDA_CHECK(cudaMemset(out, 0, sizeof(float) * nb * OUT_ELEMS));
        conv_batched_gpu(nb, x, w, bias, cols, out);
        KERNEL_CHECK();
        check("conv batched, 3 images", gpu_vs_ref(out, ref, nb * OUT_ELEMS), 1e-4);
        CUDA_CHECK(cudaMemset(out, 0, sizeof(float) * nb * OUT_ELEMS));
        conv_direct_gpu(nb, x, out);
        KERNEL_CHECK();
        check("conv direct, 3 images", gpu_vs_ref(out, ref, nb * OUT_ELEMS), 1e-4);
        dfree(x), dfree(w), dfree(bias), dfree(cols), dfree(out);
        ffree(hx), ffree(hw), ffree(hb), ffree(hcol), ffree(gcol);
        free(ref);
    }
    { // matvec: n = 37 and 1 take the scalar path, the others the float4 path
        static const int sizes[] = {1, 37, 300, 1000, 4096};
        for (int t = 0; t < 5; t++) {
            const int n = sizes[t];
            float *hA = host_random((long)n * n, 31), *hx = host_random(n, 32);
            double *ref = dalloc(n);
            float *A = to_dev(hA, (long)n * n), *x = to_dev(hx, n), *y = dzero(n);
            ref_matvec(n, hA, hx, ref);
            for (int naive = 1; naive >= 0; naive--) {
                char what[64];
                CUDA_CHECK(cudaMemset(y, 0, sizeof(float) * n));
                matvec_gpu(naive, n, A, x, y);
                KERNEL_CHECK();
                snprintf(what, sizeof what, "%-12s n=%d", naive ? "matvec_naive" : "matvec", n);
                check(what, gpu_vs_ref(y, ref, n), 1e-4);
            }
            dfree(A), dfree(x), dfree(y);
            ffree(hA), ffree(hx);
            free(ref);
        }
    }
    { // STREAM-like kernels: one length with a tail, one multiple of 4
        static const long lens[] = {1000003, 1L << 22};
        for (int t = 0; t < 2; t++) {
            const long N = lens[t];
            float *hx = host_random(N, 41), *hy = host_random(N, 42), *ho = falloc(N), got;
            double s = 0, d = 0, sabs = 0, dabs = 0;
            double *ref = dalloc(N);
            float *x = to_dev(hx, N), *y = to_dev(hy, N), *o = dzero(N), *part = dzero(g_full_blocks), *res = dzero(1);
            char what[64];
            for (long i = 0; i < N; i++) {
                s += hx[i], d += (double)hx[i] * hy[i];
                sabs += fabs(hx[i]), dabs += fabs((double)hx[i] * hy[i]);
            }
            sum_gpu(x, N, part, res);
            KERNEL_CHECK();
            to_host(&got, res, 1);
            snprintf(what, sizeof what, "sum N=%ld", N);
            check(what, fabs(got - s) / sabs, 1e-6);
            dot_gpu(x, y, N, part, res);
            KERNEL_CHECK();
            to_host(&got, res, 1);
            snprintf(what, sizeof what, "dot N=%ld", N);
            check(what, fabs(got - d) / dabs, 1e-6);
            copy_k<<<g_full_blocks, 256>>>(x, o, N);
            KERNEL_CHECK();
            to_host(ho, o, N);
            snprintf(what, sizeof what, "copy N=%ld (exact)", N);
            check(what, memcmp(hx, ho, sizeof(float) * N) ? 1.0 : 0.0, 0.0);
            triad_k<<<g_full_blocks, 256>>>(x, y, o, N);
            KERNEL_CHECK();
            for (long i = 0; i < N; i++) ref[i] = (double)hx[i] + 3.0 * hy[i];
            snprintf(what, sizeof what, "triad N=%ld", N);
            check(what, gpu_vs_ref(o, ref, N), 1e-6);
            dfree(x), dfree(y), dfree(o), dfree(part), dfree(res);
            ffree(hx), ffree(hy), ffree(ho);
            free(ref);
        }
    }
    { // FC with B vectors
        const int n = 200, B = 8;
        float *hA = host_random((long)n * n, 51), *hX = host_random((long)n * B, 52);
        double *ref = dalloc((long)n * B);
        float *A = to_dev(hA, (long)n * n), *X = to_dev(hX, (long)n * B), *Y = dzero((long)n * B);
        ref_gemm(n, B, n, hA, hX, NULL, ref);
        fc_batch_gpu(n, B, A, X, Y);
        KERNEL_CHECK();
        check("fcb n=200 B=8", gpu_vs_ref(Y, ref, (long)n * B), 1e-4);
        ref_gemm(n, 1, n, hA, hX, NULL, ref);
        fc_batch_gpu(n, 1, A, X, Y);
        KERNEL_CHECK();
        check("fcb n=200 B=1 (matvec)", gpu_vs_ref(Y, ref, n), 1e-4);
        dfree(A), dfree(X), dfree(Y);
        ffree(hA), ffree(hX);
        free(ref);
    }
    say("checks: %s (%d failed)", g_fail ? "FAILED" : "all passed", g_fail);
    return g_fail;
}

// ===================================================================== main

static int attr(cudaDeviceAttr what) {
    int v = -1;
    if (cudaDeviceGetAttribute(&v, what, 0) != cudaSuccess) {
        cudaGetLastError();
        v = -1;
    }
    return v;
}

static void write_env(const Args *a, const cudaDeviceProp &prop) {
    FILE *f = open_in(a->outdir, "env_cuda.json", "w");
    char cpu[256], ts[40], buf[128];
    int rt = 0, drv = 0;
    const int clk = attr(cudaDevAttrClockRate), mclk = attr(cudaDevAttrMemoryClockRate),
              bus = attr(cudaDevAttrGlobalMemoryBusWidth);
    cudaRuntimeGetVersion(&rt);
    cudaDriverGetVersion(&drv);
    cpu_model(cpu, sizeof cpu);
    iso_utc(ts, sizeof ts);
    fprintf(f, "{\n");
    json_str(f, "program", "bench_cuda (hand-written CUDA kernels, no cuBLAS/cuDNN)", 1);
    json_str(f, "machine", a->machine, 1);
    json_str(f, "recorded_utc", ts, 1);
    json_str(f, "cpu", cpu, 1);
    json_str(f, "gpu", prop.name, 1);
    snprintf(buf, sizeof buf, "%d.%d", prop.major, prop.minor);
    json_str(f, "gpu_compute_capability", buf, 1);
    snprintf(buf, sizeof buf, "%d.%d", rt / 1000, (rt % 1000) / 10);
    json_str(f, "cuda_runtime", buf, 1);
    snprintf(buf, sizeof buf, "%d.%d", drv / 1000, (drv % 1000) / 10);
    json_str(f, "cuda_driver_api", buf, 1);
    snprintf(buf, sizeof buf, "%d.%d.%d", __CUDACC_VER_MAJOR__, __CUDACC_VER_MINOR__, __CUDACC_VER_BUILD__);
    json_str(f, "nvcc", buf, 1);
#ifdef __VERSION__
    json_str(f, "host_compiler", __VERSION__, 1);
#endif
    json_str(f, "experiments", a->exps, 1);
    json_str(f, "batches", a->batches, 1);
    json_str(f, "direct_batches", a->direct_batches, 1);
    json_str(f, "matvec_sizes", a->mv_sizes, 1);
    json_str(f, "gemm_sizes", a->gemm_sizes, 1);
    json_str(f, "gemm_naive_sizes", a->gemm_naive_sizes, 1);
    json_str(f, "bw_sizes", a->bw_sizes, 1);
    json_str(f, "fc_sizes", a->fc_sizes, 1);
    json_str(f, "fc_b", a->fc_b, 1);
    json_str(f, "step_batches", a->step_batches, 1);
    fprintf(f, "  \"gpu_sm_count\": %d,\n  \"gpu_memory_gb\": %.1f,\n  \"gpu_max_sm_clock_mhz\": %d,\n",
            prop.multiProcessorCount, prop.totalGlobalMem / 1073741824.0, clk > 0 ? clk / 1000 : -1);
    fprintf(f, "  \"gpu_memory_clock_mhz\": %d,\n  \"gpu_memory_bus_bits\": %d,\n", mclk > 0 ? mclk / 1000 : -1, bus);
    fprintf(f, "  \"gpu_nominal_bandwidth_gbps\": %.1f,\n", (mclk > 0 && bus > 0) ? 2.0 * mclk * 1e3 * (bus / 8) / 1e9 : -1.0);
    fprintf(f, "  \"warm_seconds\": %.1f,\n  \"warmup\": %d,\n  \"repeat\": %d,\n  \"order_seed\": %d,\n  \"quick\": %d\n}\n",
            a->warm_seconds, a->warmup, a->repeat, a->order_seed, a->quick);
    fclose(f);
}

static const char *USAGE =
    "bench_cuda - hand-written CUDA benchmarks (no cuBLAS/cuDNN)\n"
    "  --check-only            run the correctness checks against float64 on the CPU and exit\n"
    "  --machine NAME          label written to every row (required unless --check-only)\n"
    "  --outdir DIR            run folder, created if needed; CSV files are appended (required)\n"
    "  --exps LIST             subset of " ALL_EXPS "\n"
    "  --warmup N --repeat N   untimed / timed runs per point (default 3 / 10)\n"
    "  --order-seed N          0 = ascending sizes, else shuffled (default 1)\n"
    "  --warm-seconds S        GPU busy time before each implementation (default 5)\n"
    "  --quick                 tiny grid, a few seconds\n"
    "  --no-check              skip the checks before timing\n"
    "  --batches, --direct-batches, --matvec-sizes, --gemm-sizes, --gemm-naive-sizes,\n"
    "  --bw-sizes, --fc-sizes, --fc-b, --step-batches LIST   comma separated grids\n";

int main(int argc, char **argv) {
    Args a;
    cudaDeviceProp prop;
    int count = 0;
    double start;
    memset(&a, 0, sizeof a);
    a.exps = ALL_EXPS;
    a.warmup = 3, a.repeat = 10, a.order_seed = 1, a.warm_seconds = 5;
    a.batches = "1,8,16,32,64,96,128,160,192,224,256";
    a.direct_batches = a.batches;
    a.mv_sizes = "256,512,1024,2048,4096,8192,12288,16384";
    a.gemm_sizes = "256,512,1024,1536,2048,3072,4096";
    a.gemm_naive_sizes = a.gemm_sizes;
    a.bw_sizes = "1024,4096,8192,16384";
    a.fc_sizes = "2048,8192";
    a.fc_b = "1,16,64,128";
    a.step_batches = "16,64,256";
    parse_args(argc, argv, &a, USAGE);

    CUDA_CHECK(cudaGetDeviceCount(&count));
    if (count < 1) die("no CUDA device");
    CUDA_CHECK(cudaSetDevice(0));
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    g_sms = prop.multiProcessorCount;
    g_full_blocks = g_sms * (prop.maxThreadsPerMultiProcessor / 256);
    if (g_full_blocks < 1) g_full_blocks = 1;
    if (g_full_blocks > 1024) g_full_blocks = 1024;
    say("GPU 0: %s, compute capability %d.%d, %d SMs, %.1f GB", prop.name, prop.major, prop.minor, g_sms,
        prop.totalGlobalMem / 1073741824.0);

    if (!a.no_check && run_checks()) return 1;
    if (a.check_only) return 0;

    mkdir_p(a.outdir);
    g_log = open_in(a.outdir, "log_cuda.txt", "a");
    write_env(&a, prop);
    say("bench_cuda: machine %s, outdir %s, warmup %d, repeat %d, order seed %d, warm %.1f s%s", a.machine, a.outdir,
        a.warmup, a.repeat, a.order_seed, a.warm_seconds, a.quick ? " (quick)" : "");
    start = now_s();
    if (has_token(a.exps, "conv")) exp_conv(&a);
    if (has_token(a.exps, "fc")) exp_fc(&a);
    if (has_token(a.exps, "gemm")) exp_gemm(&a);
    if (has_token(a.exps, "bw")) exp_bw(&a);
    if (has_token(a.exps, "fcb")) exp_fcb(&a);
    if (has_token(a.exps, "step")) exp_step(&a);
    say("done in %.1f s -> %s", now_s() - start, a.outdir);
    fclose(g_log);
    return 0;
}

In [ ]:
%%writefile /kaggle/working/c_cuda/Makefile
# Build the sequential C and the CUDA benchmarks (Linux: Kaggle, WSL Ubuntu).
#   make seq                 -> bench_seq   (needs gcc)
#   make cuda ARCH=sm_75     -> bench_cuda  (needs nvcc; sm_75 = Tesla T4, sm_60 = P100)
#   make check               -> correctness checks of both programs
# No -fopenmp, no BLAS, no cuBLAS/cuDNN: only libc, libm and the CUDA runtime are linked.

# -ffp-contract=fast lets gcc fuse a*b + c into FMA instructions. In ISO C mode (-std=c11) gcc turns this off by
# default; on Intel cores mul, add and FMA share the same ports, so without it the C code would be capped at half the
# FMA peak, while MKL and the CUDA code (fmaf) use FMA. It is not -ffast-math: no reordering of sums.
CC      = gcc
CFLAGS  = -O3 -march=native -std=c11 -ffp-contract=fast -Wall -Wextra -fno-tree-loop-distribute-patterns
NVCC    = nvcc
ARCH    ?= sm_75
NVFLAGS ?= -O3 -std=c++17 -arch=$(ARCH) -lineinfo

.PHONY: all seq cuda check clean
all: seq cuda
seq: bench_seq
cuda: bench_cuda

bench_seq: bench_seq.c common.h
	$(CC) $(CFLAGS) -o $@ bench_seq.c -lm
	@echo "$(CC) $(CFLAGS) -o $@ bench_seq.c -lm" > build_seq.txt
	@$(CC) --version | head -1 >> build_seq.txt

bench_cuda: bench_cuda.cu common.h
	$(NVCC) $(NVFLAGS) -o $@ bench_cuda.cu
	@echo "$(NVCC) $(NVFLAGS) -o $@ bench_cuda.cu" > build_cuda.txt
	@$(NVCC) --version | tail -2 >> build_cuda.txt

check: all
	./bench_seq --check-only
	./bench_cuda --check-only

clean:
	rm -f bench_seq bench_cuda build_seq.txt build_cuda.txt

In [ ]:
%%writefile /kaggle/working/c_cuda/gpu_sampler.py
"""Sample GPU 0 through NVML into a CSV until killed. Started by common.GpuClockLog.

    python gpu_sampler.py out.csv 20        # every 20 ms

A separate process, so it never competes with the benchmark for the Python GIL. Uses
ctypes on libnvidia-ml (installed with the driver), no extra package needed.
"""
import ctypes
import sys
import time


def check(code, what):
    if code != 0:
        raise RuntimeError(f"NVML {what} failed with code {code}")


def main():
    path, interval = sys.argv[1], float(sys.argv[2]) / 1000
    nvml = ctypes.CDLL("nvml.dll" if sys.platform == "win32" else "libnvidia-ml.so.1")
    check(nvml.nvmlInit_v2(), "init")
    handle = ctypes.c_void_p()
    check(nvml.nvmlDeviceGetHandleByIndex_v2(0, ctypes.byref(handle)), "handle")
    sm, mem, power, temp = (ctypes.c_uint() for _ in range(4))
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("t,sm_mhz,mem_mhz,power_w,temp_c\n")
        while True:
            check(nvml.nvmlDeviceGetClockInfo(handle, 1, ctypes.byref(sm)), "SM clock")
            check(nvml.nvmlDeviceGetClockInfo(handle, 2, ctypes.byref(mem)), "memory clock")
            # Power and temperature are extra information: 0 if the driver refuses them.
            nvml.nvmlDeviceGetPowerUsage(handle, ctypes.byref(power))
            nvml.nvmlDeviceGetTemperature(handle, 0, ctypes.byref(temp))
            fh.write(f"{time.time():.4f},{sm.value},{mem.value},{power.value / 1000:.1f},{temp.value}\n")
            fh.flush()
            time.sleep(interval)


if __name__ == "__main__":
    main()

## 3 · Helpers: run a command and stream its output (stop on failure); GPU clock log

In [ ]:
import shutil, subprocess, sys, time
from pathlib import Path
WORK = Path("/kaggle/working")
SRC, RES = WORK / "c_cuda", WORK / "results" / "kaggle"

def sh(*cmd, cwd=SRC):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"exit code {p.returncode}: {' '.join(cmd)}")

class ClockLog:
    """NVML sampler (gpu_sampler.py, every 20 ms) in its own process while the GPU program runs."""
    def __init__(self, path):
        self.path = Path(path)
    def __enter__(self):
        self.err = self.path.with_suffix(".err").open("w")
        self.proc = subprocess.Popen([sys.executable, str(SRC / "gpu_sampler.py"), str(self.path), "20"], stderr=self.err)
        time.sleep(1)
        return self
    def __exit__(self, *exc):
        self.proc.terminate()
        self.proc.wait(timeout=10)
        self.err.close()
        n = max(len(self.path.read_text().splitlines()) - 1, 0) if self.path.exists() else 0
        err = self.path.with_suffix(".err").read_text().strip()
        print(f"gpu clock log: {n} samples" + (f" | ERROR: {err.splitlines()[-1]}" if err else ""))

## 4 · Build, and show the linked libraries (no BLAS / OpenMP / cuBLAS / cuDNN)

In [ ]:
q = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap,name", "--format=csv,noheader"], capture_output=True, text=True)
first = q.stdout.strip().splitlines()[0] if q.stdout.strip() else ""
cap = first.split(",")[0].strip()
if not cap.replace(".", "").isdigit():                  # old driver without compute_cap: guess from the name
    cap = "6.0" if "P100" in first else "7.5"
ARCH = "sm_" + cap.replace(".", "")
print("GPU:", first, "-> ARCH", ARCH)
sh("make", "clean")
sh("make", "all", f"ARCH={ARCH}")
sh("ldd", "bench_seq")
sh("ldd", "bench_cuda")
print((SRC / "build_seq.txt").read_text(), (SRC / "build_cuda.txt").read_text(), sep="\n")

## 5 · Correctness checks (every kernel against float64 on the CPU)

In [ ]:
sh("./bench_seq", "--check-only")
sh("./bench_cuda", "--check-only")

## 6 · Smoke test (tiny grid, kept out of the results)

In [ ]:
shutil.rmtree(WORK / "smoke", ignore_errors=True)
with ClockLog(WORK / "smoke_clocks.csv"):
    sh("./bench_cuda", "--machine", "kaggle", "--outdir", WORK / "smoke", "--quick", "--no-check")
sh("./bench_seq", "--machine", "kaggle", "--outdir", WORK / "smoke", "--quick", "--no-check")

## 7 · 3 runs: GPU first (30 repeats, NVML clocks), then CPU (10 repeats); shuffled sizes

In [ ]:
from datetime import datetime
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RES.mkdir(parents=True, exist_ok=True)
sh("bash", "-c", f"lscpu > {RES}/lscpu_c_cuda_{STAMP}.txt")
T0 = time.time()

def one_run(i):
    run = RES / f"c_cuda{i}_{STAMP}"
    run.mkdir(parents=True)
    for name in ("build_seq.txt", "build_cuda.txt"):
        shutil.copy(SRC / name, run / name)
    with ClockLog(run / "gpu_clocks.csv"):
        sh("./bench_cuda", "--machine", "kaggle", "--outdir", run, "--order-seed", i, "--repeat", 30)
    sh("./bench_seq", "--machine", "kaggle", "--outdir", run, "--order-seed", i, "--repeat", 10)
    print(f"run {i} done, {(time.time() - T0) / 60:.1f} min since the first run started")
    return run

In [ ]:
RUN1 = one_run(1)

In [ ]:
RUN2 = one_run(2)

In [ ]:
RUN3 = one_run(3)

## 8 · Quick look (medians over the runs; the analysis is done offline)

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
def load(name):
    return pd.concat([pd.read_csv(r / name).assign(run=r.name[:7]) for r in (RUN1, RUN2, RUN3)], ignore_index=True)
conv = load("conv.csv"); conv["ms_per_img"] = conv["mean_ms"] / conv["size"]
print("conv, ms per image"); print(conv.pivot_table(index=["device", "impl"], columns="size", values="ms_per_img", aggfunc="median").round(4).to_string())
fc = load("fc.csv")
mv, gm = fc[fc.impl.str.startswith("matvec")], fc[fc.impl.str.startswith("gemm")]
print("\nmatvec, GB/s"); print(mv.pivot_table(index=["device", "impl"], columns="size", values="gbps", aggfunc="median").round(1).to_string())
print("\ngemm, GFLOP/s"); print(gm.pivot_table(index=["device", "impl"], columns="size", values="gflops", aggfunc="median").round(0).to_string())
bw = load("bw.csv")
print("\nbandwidth, GB/s"); print(bw.pivot_table(index=["device", "impl"], columns="size", values="gbps", aggfunc="median").round(1).to_string())

## 9 · Package the results for download

In [ ]:
print(shutil.make_archive(str(WORK / "results_c_cuda"), "zip", WORK, "results"))